# Why does a cardiac MRI diagnosis model fail at a new hospital?
## Measurement-level uncertainty and failure attribution under multi-vendor shift

This notebook follows the paper section by section. Each part starts with the question it answers and ends with the numbers that go into the manuscript, so the code can be read in the same order as the text.

**The story in one line.** A simple rule on 21 standard measurements (Part 2) is compared with stronger models (Part 3); each patient's measurement uncertainty gives a flag that names the doubtful measurement (Part 4); on a multi-vendor external cohort, every error is traced to segmentation or to the rule (Part 5); a few labelled cases re-anchor the rule (Part 6); ablations and robustness checks test every choice (Part 7); two ideas that failed are reported (Part 8); figures close the notebook (Part 9).

**How to run.** In order, top to bottom. Every step reads and writes one Drive folder (`ACDC_DRIVE`), so a dropped session resumes from the last finished step. Steps marked **GPU** need a GPU runtime (one T4); the others run on CPU.

**Leakage rules (every step).** (1) Each patient is segmented by the U-Net of the fold that held it out. (2) Scalers, classifiers, regularisation, κ and conformal thresholds are fitted on training patients only. (3) The rule is fitted on reference measurements of training patients and applied to predicted measurements of held-out patients. (4) Everything used on M&Ms-1 is frozen on ACDC first; the only post hoc change (ED/ES ordering) is reported in both versions.

| Part | Paper | Steps |
|---|---|---|
| 1. Data | 3.1 | setup, folds, M&Ms-1 inspection, data inventory |
| 2. Measurements and decision rule | 3.2–3.3 | U-Nets, 21 measurements, logistic rule |
| 3. Comparators | 3.4, 4.1 | random forest, CineMA, joint model, additive model |
| 4. Uncertainty and the flip flag | 3.5–3.7, 4.2 | stochastic segmentations, κ, flag, bootstrap |
| 5. External test on M&Ms-1 | 3.8, 4.3–4.4 | locked test, error attribution, ED/ES rule, DCM strata, other classes |
| 6. Few-shot recalibration | 3.9, 4.5 | class offsets vs full refit |
| 7. Ablation and robustness | 3.10, 4.6 | ablations and paired tests, flag threshold, repeated CV, BSA |
| 8. Ideas that did not work | 4.7 | physiological laws, single error model |
| 9. Figures | Figs 2–5 | all figures |

## Part 1 — Data (Section 3.1)
*Which patients, which labels, which folds?* ACDC (100 patients, 5 classes) is used for development with five patient-level folds; M&Ms-1 is used only for the locked external test. The inventory checks what the external data allow: labels per centre and height/weight for body-surface-area indexing.

### Setup
Mount Drive, set the two paths used by every step, unzip ACDC once.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install nibabel

import os, glob, zipfile
os.environ['ACDC_DRIVE'] = '/content/drive/MyDrive/CMR_concept_bottleneck'
os.environ['ACDC_ROOT'] = '/content/acdc/training'
os.makedirs(os.environ['ACDC_DRIVE'], exist_ok=True)
if not os.path.exists(f"{os.environ['ACDC_ROOT']}/patient001/Info.cfg"):
    acdc_zip = next(p for p in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True)
                    if any(n.endswith('patient001/Info.cfg') for n in zipfile.ZipFile(p).namelist()))
    zipfile.ZipFile(acdc_zip).extractall('/content/acdc_raw')
    first = glob.glob('/content/acdc_raw/**/patient001/Info.cfg', recursive=True)[0]
    os.makedirs('/content/acdc', exist_ok=True)
    os.symlink(os.path.dirname(os.path.dirname(first)), os.environ['ACDC_ROOT'])
print('ACDC patients:', len(glob.glob(f"{os.environ['ACDC_ROOT']}/patient*/Info.cfg")))


### Patient-level folds
Five stratified folds (seed 42), written once to `folds.csv` and shared by every step.

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
FOLDS = f"{os.environ['ACDC_DRIVE']}/folds.csv"
if not os.path.exists(FOLDS):
    rows = []
    for cfg in sorted(glob.glob(f"{os.environ['ACDC_ROOT']}/patient*/Info.cfg")):
        info = dict(l.split(':', 1) for l in open(cfg) if ':' in l)
        rows.append(dict(patient=os.path.basename(os.path.dirname(cfg)), label=info['Group'].strip()))
    df = pd.DataFrame(rows)
    assert len(df) == 100 and (df.label.value_counts() == 20).all()
    df['fold'] = -1
    for k, (_, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(df, df.label)):
        df.loc[te, 'fold'] = k
    df.to_csv(FOLDS, index=False)
print(pd.read_csv(FOLDS).groupby(['fold', 'label']).size().unstack())


### M&Ms-1 inspection (CPU, read-only)
Layout, metadata and label conventions of the M&Ms-1 release.

In [ ]:
import os, glob, zipfile, io, collections
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive/CMR_concept_bottleneck'
cands = glob.glob('/content/drive/MyDrive/**/M&M_dataset.zip', recursive=True)
print('zip found at:', cands)
assert cands, 'Add a shortcut to M&M_dataset.zip in My Drive (CMR_concept_bottleneck) first'
ZIP = cands[0]

z = zipfile.ZipFile(ZIP)
names = z.namelist()
print('entries in zip:', len(names))

tree = collections.Counter('/'.join(n.split('/')[:3]) for n in names)
print('\nTop of the layout (path : number of entries):')
for k, v in sorted(tree.items())[:60]:
    print(f'  {k} : {v}')

ext = collections.Counter(os.path.splitext(n)[1].lower() if not n.endswith('.gz') else '.nii.gz' for n in names)
print('\nFile types:', dict(ext))

subj = collections.defaultdict(set)
for n in names:
    parts = n.strip('/').split('/')
    if n.endswith('.nii.gz') and len(parts) >= 3:
        subj['/'.join(parts[:-2])].add(parts[-2])
print('\nSubjects per folder:')
for k, v in sorted(subj.items()):
    print(f'  {k}: {len(v)} subjects, e.g. {sorted(v)[:2]}')

ex = [n for n in names if n.endswith('.nii.gz')][:6]
print('\nExample image files:', ex)

meta = [n for n in names if n.lower().endswith(('.csv', '.xlsx', '.xls'))]
print('\nMetadata files:', meta)
for m in meta:
    raw = z.read(m)
    df = pd.read_csv(io.BytesIO(raw)) if m.lower().endswith('.csv') else pd.read_excel(io.BytesIO(raw))
    print(f'\n=== {m}: {df.shape}')
    print('columns:', list(df.columns))
    print(df.head(3).to_string())
    for col in df.columns:
        if any(w in col.lower() for w in ('vendor', 'centre', 'center', 'patholog', 'diagnos', 'group')):
            print(f'\n  value counts of {col}:')
            print(df[col].value_counts(dropna=False).to_string())


### Data inventory (CPU)
Labels per centre and height/weight availability; writes ACDC height and weight for Part 7.

In [ ]:
import os, io, re, glob, zipfile, pandas as pd
D = os.environ['ACDC_DRIVE']; os.makedirs(f'{D}/step14_robustness', exist_ok=True)
zm = zipfile.ZipFile(glob.glob('/content/drive/MyDrive/**/M&M*.zip', recursive=True)[0])
meta = pd.read_csv(io.BytesIO(zm.read([n for n in zm.namelist() if 'MnMs_Dataset_information' in n][0])))
meta['External code'] = meta['External code'].astype(str).str.strip()
gt_codes = {re.search(r'([A-Z0-9]{6})_sa_gt\.nii', n).group(1) for n in zm.namelist() if re.search(r'[A-Z0-9]{6}_sa_gt\.nii', n)}
m = meta[meta['External code'].isin(gt_codes)]
print(f'M&Ms-1 subjects with contours: {len(m)}'); print(pd.crosstab(m['Pathology'], m['Centre'], margins=True).to_string())
has = (pd.to_numeric(m['Height'], errors='coerce') > 0) & (pd.to_numeric(m['Weight'], errors='coerce') > 0)
print(f'Height and weight available: {has.sum()}/{len(m)}'); print(pd.crosstab(m['Centre'], has, colnames=['height+weight']).to_string())
rows = []
for cfg in sorted(glob.glob(f"{os.environ['ACDC_ROOT']}/patient*/Info.cfg")):
    info = dict(l.split(':', 1) for l in open(cfg) if ':' in l)
    rows.append(dict(patient=os.path.basename(os.path.dirname(cfg)), height=float(info['Height']), weight=float(info['Weight'])))
hw = pd.DataFrame(rows); hw.to_csv(f'{D}/step14_robustness/acdc_height_weight.csv', index=False)
print(f'ACDC patients with height and weight: {((hw.height > 0) & (hw.weight > 0)).sum()}/{len(hw)}')


## Part 2 — Measurements and decision rule (Sections 3.2–3.3)
*Can a linear rule on standard measurements diagnose well?* One U-Net per fold segments the held-out patients; 21 measurements are computed from every mask; the rule is fitted on reference measurements and applied to predicted ones. Step 3 also fits the challenge-style random forest used in Part 3.

### U-Nets, one per fold (GPU, ~5–10 min per fold)
Epoch chosen on inner-validation patients only.

In [ ]:
import os, glob, zipfile, time, copy
import numpy as np, pandas as pd, nibabel as nib
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.ndimage import distance_transform_edt
from sklearn.model_selection import StratifiedShuffleSplit

try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, NotImplementedError):
    pass

KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if KAGGLE:
    DRIVE = '/kaggle/working'
    ROOT = os.path.dirname(os.path.dirname(sorted(glob.glob('/kaggle/input/**/patient001/Info.cfg', recursive=True))[0]))
    FOLDS = sorted(glob.glob('/kaggle/input/**/folds.csv', recursive=True))[0]
else:
    DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
    ROOT = os.environ.get('ACDC_ROOT', '/content/acdc/training')
    FOLDS = f'{DRIVE}/folds.csv'
ZIP = '/content/drive/MyDrive/ACDC_calibration_article/training-20261021T163705Z-1-001.zip'
OUT = f"{DRIVE}/step1_unet{os.environ.get('RUN_TAG', '')}"
os.makedirs(OUT, exist_ok=True)

SEED, BS, LR = 42, 16, 1e-3
SIZE, WIDTH, EPOCHS = (int(os.environ.get(k, d)) for k, d in (('SIZE', 224), ('WIDTH', 32), ('EPOCHS', 40)))
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(os.cpu_count() or 1)
print(f'device: {dev} | SIZE {SIZE} | WIDTH {WIDTH} | EPOCHS {EPOCHS} | data {ROOT} | out {OUT}')

if not os.path.isdir(ROOT):
    print('unzipping ACDC ...')
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content/acdc')
pids_all = sorted(os.path.basename(p) for p in glob.glob(f'{ROOT}/patient*'))
print('patients found:', len(pids_all))

def read_cfg(p):
    d = {}
    for line in open(p):
        if ':' in line:
            k, v = line.split(':', 1)
            d[k.strip()] = v.strip()
    return d

def norm(v):
    lo, hi = np.percentile(v, [1, 99])
    v = np.clip(v, lo, hi)
    return ((v - v.mean()) / (v.std() + 1e-8)).astype(np.float32)

def nii(base):
    for e in ('.nii.gz', '.nii'):
        if os.path.exists(base + e):
            return base + e
    raise FileNotFoundError(base + '.nii[.gz]')

def load_patient(pid):
    d = f'{ROOT}/{pid}'
    info = read_cfg(f'{d}/Info.cfg')
    out = {'info': info}
    for ph in ('ED', 'ES'):
        fr = int(info[ph])
        img = nib.load(nii(f'{d}/{pid}_frame{fr:02d}'))
        gt = nib.load(nii(f'{d}/{pid}_frame{fr:02d}_gt'))
        out[ph] = (norm(img.get_fdata()),
                   np.rint(gt.get_fdata()).astype(np.int64),
                   tuple(float(z) for z in img.header.get_zooms()[:3]))
    return out

t0 = time.time()
P = {p: load_patient(p) for p in pids_all}
print(f'loaded in {time.time() - t0:.0f}s')

def rs(a, mode):
    t = torch.from_numpy(np.ascontiguousarray(a))[None, None].float()
    if mode == 'nearest':
        return F.interpolate(t, size=(SIZE, SIZE), mode='nearest')[0, 0]
    return F.interpolate(t, size=(SIZE, SIZE), mode='bilinear', align_corners=False)[0, 0]

def slices(pids):
    X, Y = [], []
    for p in pids:
        for ph in ('ED', 'ES'):
            v, g, _ = P[p][ph]
            for z in range(v.shape[2]):
                X.append(rs(v[:, :, z], 'bilinear'))
                Y.append(rs(g[:, :, z], 'nearest').long())
    return torch.stack(X)[:, None], torch.stack(Y)

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(s, c=32, n=4, p=0.1):
        super().__init__()
        s.e1, s.e2, s.e3, s.e4 = block(1, c), block(c, 2 * c), block(2 * c, 4 * c), block(4 * c, 8 * c)
        s.b = block(8 * c, 16 * c)
        s.drop = nn.Dropout2d(p)
        s.u4, s.d4 = nn.ConvTranspose2d(16 * c, 8 * c, 2, 2), block(16 * c, 8 * c)
        s.u3, s.d3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, 2), block(8 * c, 4 * c)
        s.u2, s.d2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, 2), block(4 * c, 2 * c)
        s.u1, s.d1 = nn.ConvTranspose2d(2 * c, c, 2, 2), block(2 * c, c)
        s.out = nn.Conv2d(c, n, 1)

    def forward(s, x):
        e1 = s.e1(x)
        e2 = s.e2(F.max_pool2d(e1, 2))
        e3 = s.e3(F.max_pool2d(e2, 2))
        e4 = s.e4(F.max_pool2d(e3, 2))
        b = s.drop(s.b(F.max_pool2d(e4, 2)))
        d4 = s.d4(torch.cat([s.u4(b), e4], 1))
        d3 = s.d3(torch.cat([s.u3(d4), e3], 1))
        d2 = s.d2(torch.cat([s.u2(d3), e2], 1))
        d1 = s.d1(torch.cat([s.u1(d2), e1], 1))
        return s.out(d1)

def dice_loss(logits, y):
    p = logits.softmax(1)
    oh = F.one_hot(y, 4).permute(0, 3, 1, 2).float()
    inter = (p * oh).sum((0, 2, 3))
    den = p.sum((0, 2, 3)) + oh.sum((0, 2, 3))
    return 1 - ((2 * inter + 1) / (den + 1))[1:].mean()

def aug(x, y):
    if torch.rand(1) < 0.5:
        x, y = x.flip(-1), y.flip(-1)
    if torch.rand(1) < 0.5:
        x, y = x.flip(-2), y.flip(-2)
    k = int(torch.randint(0, 4, (1,)))
    x, y = torch.rot90(x, k, (-2, -1)), torch.rot90(y, k, (-2, -1))
    n = x.size(0)
    x = x * (1 + 0.1 * torch.randn(n, 1, 1, 1, device=x.device)) + 0.1 * torch.randn(n, 1, 1, 1, device=x.device)
    return x, y

@torch.no_grad()
def evaluate(net, X, Y):
    net.eval()
    L, inter, den = 0.0, np.zeros(3), np.zeros(3)
    for b in range(0, len(X), 64):
        x, y = X[b:b + 64].to(dev), Y[b:b + 64].to(dev)
        lo = net(x).float()
        L += F.cross_entropy(lo, y, reduction='none').mean((1, 2)).sum().item()
        pr = lo.argmax(1)
        for c in (1, 2, 3):
            inter[c - 1] += ((pr == c) & (y == c)).sum().item()
            den[c - 1] += (pr == c).sum().item() + (y == c).sum().item()
    return L / len(X), 2 * inter / np.maximum(den, 1)

@torch.no_grad()
def predict_probs(net, v):
    """v: (H, W, Z) normalised volume -> probs (4, H, W, Z) at native resolution."""
    net.eval()
    H, W, Z = v.shape
    x = torch.stack([rs(v[:, :, z], 'bilinear') for z in range(Z)])[:, None].to(dev)
    p = net(x).float().softmax(1)
    p = F.interpolate(p, size=(H, W), mode='bilinear', align_corners=False)
    return p.permute(1, 2, 3, 0).cpu().numpy()

def wall_thickness_mm(myo, zooms):
    best = 0.0
    for z in range(myo.shape[2]):
        m = myo[:, :, z]
        if m.sum() < 10:
            continue
        best = max(best, 2 * distance_transform_edt(m, sampling=zooms[:2]).max() - np.mean(zooms[:2]))
    return best

def biomarkers(prob_ed, prob_es, zooms_ed, zooms_es, info):
    """prob_*: (4, H, W, Z) soft probabilities or one-hot. Classes 1 RV, 2 MYO, 3 LV."""
    ved, ves = np.prod(zooms_ed) / 1000.0, np.prod(zooms_es) / 1000.0
    lv_edv, lv_esv = prob_ed[3].sum() * ved, prob_es[3].sum() * ves
    rv_edv, rv_esv = prob_ed[1].sum() * ved, prob_es[1].sum() * ves
    myo_ed = prob_ed[2].sum() * ved
    h, w = float(info.get('Height', 'nan')), float(info.get('Weight', 'nan'))
    bsa = np.sqrt(h * w / 3600.0)
    return dict(LV_EDV=lv_edv, LV_ESV=lv_esv, LV_EF=100 * (lv_edv - lv_esv) / max(lv_edv, 1e-6),
                RV_EDV=rv_edv, RV_ESV=rv_esv, RV_EF=100 * (rv_edv - rv_esv) / max(rv_edv, 1e-6),
                LV_mass=1.05 * myo_ed,
                MaxWT=wall_thickness_mm(prob_ed[2] > 0.5, zooms_ed),
                LV_EDVi=lv_edv / bsa, RV_EDVi=rv_edv / bsa, LV_massi=1.05 * myo_ed / bsa,
                LV_RV_EDV_ratio=lv_edv / max(rv_edv, 1e-6), BSA=bsa)

def onehot(lbl):
    return np.stack([(lbl == c).astype(np.float32) for c in range(4)])

def dice3d(pred, gt, c):
    a, b = pred == c, gt == c
    s = a.sum() + b.sum()
    return 2 * (a & b).sum() / s if s else np.nan

folds = pd.read_csv(FOLDS)
lab = dict(zip(folds.patient, folds.label))
fold_of = dict(zip(folds.patient, folds.fold))
assert set(folds.patient) == set(pids_all), 'folds.csv and data folder do not match'

bio_rows, dice_rows, curve_rows, split_rows = [], [], [], []

for p in pids_all:
    ed, es = P[p]['ED'], P[p]['ES']
    bio_rows.append(dict(patient=p, fold=fold_of[p], label=lab[p], source='gt',
                         **biomarkers(onehot(ed[1]), onehot(es[1]), ed[2], es[2], P[p]['info'])))

for k in range(5):
    t0 = time.time()
    tr_all = folds.patient[folds.fold != k].values
    te = folds.patient[folds.fold == k].values
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
    i_tr, i_va = next(sss.split(tr_all, [lab[p] for p in tr_all]))
    tr, va = tr_all[i_tr], tr_all[i_va]
    assert not (set(tr) & set(te)) and not (set(va) & set(te)) and not (set(tr) & set(va))
    for role, ps in (('train', tr), ('inner_val', va), ('test', te)):
        split_rows += [dict(fold=k, role=role, patient=p) for p in ps]

    Xtr, Ytr = slices(tr)
    Xva, Yva = slices(va)
    torch.manual_seed(SEED + k)
    net = UNet(c=WIDTH).to(dev)
    opt = torch.optim.AdamW(net.parameters(), LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    use_amp = dev == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    best, best_ep = -1.0, 0
    for ep in range(EPOCHS):
        net.train()
        perm = torch.randperm(len(Xtr))
        tl, seen = 0.0, 0
        for b in range(0, len(perm) - BS + 1, BS):
            idx = perm[b:b + BS]
            x, y = aug(Xtr[idx].to(dev), Ytr[idx].to(dev))
            with torch.autocast(device_type=dev, enabled=use_amp):
                lo = net(x)
            loss = F.cross_entropy(lo.float(), y) + dice_loss(lo.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tl += loss.item() * len(idx)
            seen += len(idx)
        sched.step()
        vl, vd = evaluate(net, Xva, Yva)
        curve_rows.append(dict(fold=k, epoch=ep + 1, train_loss=tl / max(seen, 1), val_ce=vl,
                               val_dice_RV=vd[0], val_dice_MYO=vd[1], val_dice_LV=vd[2], val_dice_mean=vd.mean()))
        if vd.mean() > best:
            best, best_ep = vd.mean(), ep + 1
            best_state = copy.deepcopy(net.state_dict())

    net.load_state_dict(best_state)
    torch.save(best_state, f'{OUT}/unet_fold{k}.pt')
    for p in te:
        ed, es = P[p]['ED'], P[p]['ES']
        pe, ps_ = predict_probs(net, ed[0]), predict_probs(net, es[0])
        he, hs = pe.argmax(0), ps_.argmax(0)
        bio_rows.append(dict(patient=p, fold=k, label=lab[p], source='pred_soft',
                             **biomarkers(pe, ps_, ed[2], es[2], P[p]['info'])))
        bio_rows.append(dict(patient=p, fold=k, label=lab[p], source='pred_hard',
                             **biomarkers(onehot(he), onehot(hs), ed[2], es[2], P[p]['info'])))
        dice_rows.append(dict(patient=p, fold=k, label=lab[p],
                              **{f'{ph}_{n}': dice3d(h, g[1], c)
                                 for ph, h, g in (('ED', he, ed), ('ES', hs, es))
                                 for n, c in (('RV', 1), ('MYO', 2), ('LV', 3))}))
    print(f'fold {k}: best epoch {best_ep}/{EPOCHS}, inner-val Dice {best:.3f}, {time.time() - t0:.0f}s')
    pd.DataFrame(curve_rows).to_csv(f'{OUT}/learning_curves.csv', index=False)

pd.DataFrame(split_rows).to_csv(f'{OUT}/split_check.csv', index=False)
pd.DataFrame(bio_rows).to_csv(f'{OUT}/biomarkers.csv', index=False)
dice = pd.DataFrame(dice_rows)
dice.to_csv(f'{OUT}/dice_test.csv', index=False)

print('\nHeld-out 3D Dice (mean over 100 patients):')
print(dice.drop(columns=['patient', 'fold', 'label']).mean().round(3).to_string())
print('\nSaved to', OUT)


### 21 measurements and random forest (CPU/GPU, ~20 min)
Reference and predicted measurements for every patient; predicted masks are saved for later parts.

In [ ]:
import os, glob, zipfile, time
import numpy as np, pandas as pd, nibabel as nib, torch, torch.nn as nn, torch.nn.functional as F
from scipy.ndimage import distance_transform_edt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, f1_score, confusion_matrix

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
STEP1 = os.environ.get('STEP1_DIR', f'{DRIVE}/step1_unet')
OUT = f'{DRIVE}/step3_challenge_baseline'
ROOT = os.environ.get('ACDC_ROOT', '/content/acdc/training')
os.makedirs(f'{OUT}/pred_masks', exist_ok=True)
if not os.path.isdir(ROOT):
    zipfile.ZipFile('/content/drive/MyDrive/ACDC_calibration_article/training-20261021T163705Z-1-001.zip').extractall('/content/acdc')
SEED, SIZE, WIDTH, B = 42, int(os.environ.get('SIZE', 224)), int(os.environ.get('WIDTH', 32)), 2000
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(os.cpu_count() or 1)

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(s, c=32, n=4, p=0.1):
        super().__init__()
        s.e1, s.e2, s.e3, s.e4 = block(1, c), block(c, 2 * c), block(2 * c, 4 * c), block(4 * c, 8 * c)
        s.b = block(8 * c, 16 * c)
        s.drop = nn.Dropout2d(p)
        s.u4, s.d4 = nn.ConvTranspose2d(16 * c, 8 * c, 2, 2), block(16 * c, 8 * c)
        s.u3, s.d3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, 2), block(8 * c, 4 * c)
        s.u2, s.d2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, 2), block(4 * c, 2 * c)
        s.u1, s.d1 = nn.ConvTranspose2d(2 * c, c, 2, 2), block(2 * c, c)
        s.out = nn.Conv2d(c, n, 1)

    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(F.max_pool2d(e1, 2)); e3 = s.e3(F.max_pool2d(e2, 2)); e4 = s.e4(F.max_pool2d(e3, 2))
        b = s.drop(s.b(F.max_pool2d(e4, 2)))
        d4 = s.d4(torch.cat([s.u4(b), e4], 1)); d3 = s.d3(torch.cat([s.u3(d4), e3], 1))
        d2 = s.d2(torch.cat([s.u2(d3), e2], 1)); d1 = s.d1(torch.cat([s.u1(d2), e1], 1))
        return s.out(d1)

def read_cfg(p):
    return {k.strip(): v.strip() for k, v in (l.split(':', 1) for l in open(p) if ':' in l)}

def nii(base):
    for e in ('.nii.gz', '.nii'):
        if os.path.exists(base + e):
            return base + e
    raise FileNotFoundError(base)

def norm(v):
    lo, hi = np.percentile(v, [1, 99])
    v = np.clip(v, lo, hi)
    return ((v - v.mean()) / (v.std() + 1e-8)).astype(np.float32)

@torch.no_grad()
def predict_mask(net, v):
    H, W, Z = v.shape
    x = torch.from_numpy(np.ascontiguousarray(v.transpose(2, 0, 1)))[:, None].to(dev)
    x = F.interpolate(x, size=(SIZE, SIZE), mode='bilinear', align_corners=False)
    p = F.interpolate(net(x).float().softmax(1), size=(H, W), mode='bilinear', align_corners=False)
    return p.argmax(1).permute(1, 2, 0).cpu().numpy().astype(np.uint8)

def thickness_per_slice(myo, zooms):
    out = []
    for z in range(myo.shape[2]):
        m = myo[:, :, z]
        if m.sum() >= 10:
            out.append(2 * distance_transform_edt(m, sampling=zooms[:2]).max() - np.mean(zooms[:2]))
    return np.array(out) if out else np.array([0.0])

def rich_features(lab_ed, lab_es, z_ed, z_es):
    ved, ves = np.prod(z_ed) / 1000, np.prod(z_es) / 1000
    v = {f'{s}_{ph}': (lab == c).sum() * vv for ph, lab, vv in (('ED', lab_ed, ved), ('ES', lab_es, ves))
         for s, c in (('RV', 1), ('MYO', 2), ('LV', 3))}
    t_ed, t_es = thickness_per_slice(lab_ed == 2, z_ed), thickness_per_slice(lab_es == 2, z_es)
    eps = 1e-6
    return dict(**v,
                LV_EF=100 * (v['LV_ED'] - v['LV_ES']) / max(v['LV_ED'], eps),
                RV_EF=100 * (v['RV_ED'] - v['RV_ES']) / max(v['RV_ED'], eps),
                LV_mass=1.05 * v['MYO_ED'],
                RV_LV_ED=v['RV_ED'] / max(v['LV_ED'], eps), RV_LV_ES=v['RV_ES'] / max(v['LV_ES'], eps),
                MYO_LV_ED=v['MYO_ED'] / max(v['LV_ED'], eps), MYO_LV_ES=v['MYO_ES'] / max(v['LV_ES'], eps),
                WT_ED_max=t_ed.max(), WT_ED_mean=t_ed.mean(), WT_ED_std=t_ed.std(), WT_ED_min=t_ed.min(),
                WT_ES_max=t_es.max(), WT_ES_mean=t_es.mean(), WT_ES_std=t_es.std(), WT_ES_min=t_es.min())

folds = pd.read_csv(f'{DRIVE}/folds.csv')
order = folds.patient.values
classes = np.array(sorted(folds.label.unique()))
y = np.searchsorted(classes, folds.label.values)
f = folds.fold.values
ref, pred = {}, {}
t0 = time.time()
for k in range(5):
    net = UNet(c=WIDTH).to(dev)
    net.load_state_dict(torch.load(f'{STEP1}/unet_fold{k}.pt', map_location=dev))
    net.eval()
    for p in order[f == k]:
        d = f'{ROOT}/{p}'
        info = read_cfg(f'{d}/Info.cfg')
        labs, gts, zs = [], [], []
        for ph in ('ED', 'ES'):
            img = nib.load(nii(f'{d}/{p}_frame{int(info[ph]):02d}'))
            gts.append(np.rint(nib.load(nii(f'{d}/{p}_frame{int(info[ph]):02d}_gt')).get_fdata()).astype(np.uint8))
            zs.append(tuple(float(z) for z in img.header.get_zooms()[:3]))
            labs.append(predict_mask(net, norm(img.get_fdata())))
        np.savez_compressed(f'{OUT}/pred_masks/{p}.npz', ED=labs[0], ES=labs[1], zooms_ED=zs[0], zooms_ES=zs[1])
        pred[p] = rich_features(labs[0], labs[1], zs[0], zs[1])
        ref[p] = rich_features(gts[0], gts[1], zs[0], zs[1])
    print(f'fold {k}: masks predicted ({time.time() - t0:.0f}s)')

R = pd.DataFrame.from_dict(ref, orient='index').loc[order]
Q = pd.DataFrame.from_dict(pred, orient='index').loc[order]
FEATS = list(R.columns)
pd.concat([R.assign(source='gt'), Q.assign(source='pred')]).rename_axis('patient').to_csv(f'{OUT}/rich_features.csv')
print(len(FEATS), 'features:', FEATS)

P = {'challenge_RF_pred': np.zeros((len(y), len(classes))), 'challenge_RF_GT_upper': np.zeros((len(y), len(classes)))}
for k in range(5):
    tr, te = f != k, f == k
    grid = GridSearchCV(RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=SEED, n_jobs=-1),
                        {'max_features': ['sqrt', 0.5], 'min_samples_leaf': [1, 3]},
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy')
    grid.fit(R.values[tr], y[tr])
    P['challenge_RF_pred'][te] = grid.predict_proba(Q.values[te])
    P['challenge_RF_GT_upper'][te] = grid.predict_proba(R.values[te])
    print(f'fold {k}: {grid.best_params_}')

for name, path in (('concept_pred_soft (ours)', f'{STEP1}/oof_probs_concept_pred_soft.npy'),
                   ('cinema_frozen_probe', f'{DRIVE}/cinema_oof_probs.npy')):
    if os.path.exists(path):
        P[name] = np.load(path)

def ece(Pm, yy, bins=15):
    conf, acc = Pm.max(1), (Pm.argmax(1) == yy).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    return sum(((conf > lo) & (conf <= hi)).mean() * abs(acc[(conf > lo) & (conf <= hi)].mean() - conf[(conf > lo) & (conf <= hi)].mean())
               for lo, hi in zip(edges[:-1], edges[1:]) if ((conf > lo) & (conf <= hi)).any())

def metrics(Pm, i):
    return (balanced_accuracy_score(y[i], Pm[i].argmax(1)),
            roc_auc_score(y[i], Pm[i], multi_class='ovr', average='macro', labels=range(len(classes))), ece(Pm[i], y[i]))

rng = np.random.default_rng(SEED)
boots = [i for i in (rng.integers(0, len(y), len(y)) for _ in range(B)) if len(np.unique(y[i])) == len(classes)]
rows = []
for name, Pm in P.items():
    pt, bs = metrics(Pm, np.arange(len(y))), np.array([metrics(Pm, i) for i in boots])
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
    rows.append(dict(model=name, bal_acc=pt[0], bal_lo=lo[0], bal_hi=hi[0], AUC=pt[1], AUC_lo=lo[1], AUC_hi=hi[1],
                     ECE=pt[2], macro_F1=f1_score(y, Pm.argmax(1), average='macro')))
res = pd.DataFrame(rows)
print('\n', res.round(3).to_string(index=False))

ours = 'concept_pred_soft (ours)'
if ours in P:
    print(f'\nPaired difference: {ours} minus challenge_RF_pred')
    d = np.array([np.subtract(metrics(P[ours], i)[:2], metrics(P['challenge_RF_pred'], i)[:2]) for i in boots])
    for j, m in enumerate(['bal_acc', 'AUC']):
        print(f'  {m}: {d[:, j].mean():+.3f}  95% CI [{np.percentile(d[:, j], 2.5):+.3f}, {np.percentile(d[:, j], 97.5):+.3f}]')

print('\nConfusion matrix, challenge_RF_pred (rows = truth):', [str(c) for c in classes])
print(confusion_matrix(y, P['challenge_RF_pred'].argmax(1)))
res.to_csv(f'{OUT}/step3_results.csv', index=False)
for name in ('challenge_RF_pred', 'challenge_RF_GT_upper'):
    np.save(f'{OUT}/oof_probs_{name}.npy', P[name])
print('\nSaved to', OUT)


### Logistic rule — the main model (CPU)
Row 'Logistic rule' of Table 1.

In [ ]:
import os, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, f1_score, confusion_matrix

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
S3 = f'{DRIVE}/step3_challenge_baseline'
OUT = f'{DRIVE}/step2b_concepts21'
os.makedirs(OUT, exist_ok=True)
SEED, B = 42, 2000

folds = pd.read_csv(f'{DRIVE}/folds.csv')
order = folds.patient.values
rf = pd.read_csv(f'{S3}/rich_features.csv')
FEATS = [c for c in rf.columns if c not in ('patient', 'source')]
R = rf[rf.source == 'gt'].set_index('patient').loc[order, FEATS].values
Q = rf[rf.source == 'pred'].set_index('patient').loc[order, FEATS].values
classes = np.array(sorted(folds.label.unique()))
y = np.searchsorted(classes, folds.label.values)
f = folds.fold.values
print(len(FEATS), 'concepts')

P = {'concept21_LR_pred (ours)': np.zeros((len(y), len(classes))),
     'concept21_LR_GT_upper': np.zeros((len(y), len(classes)))}
coefs = []
for k in range(5):
    tr, te = f != k, f == k
    grid = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced')),
                        {'logisticregression__C': [0.01, 0.1, 1, 10, 100]},
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy')
    grid.fit(R[tr], y[tr])
    P['concept21_LR_pred (ours)'][te] = grid.predict_proba(Q[te])
    P['concept21_LR_GT_upper'][te] = grid.predict_proba(R[te])
    lr = grid.best_estimator_.named_steps['logisticregression']
    coefs += [dict(fold=k, cls=str(classes[i]), **dict(zip(FEATS, lr.coef_[i]))) for i in range(len(classes))]
    print(f'fold {k}: C = {grid.best_params_["logisticregression__C"]}')

others = {'challenge_RF_pred': f'{S3}/oof_probs_challenge_RF_pred.npy',
          'concept7_LR_pred (previous)': f'{DRIVE}/step1_unet/oof_probs_concept_pred_soft.npy',
          'cinema_frozen_probe': f'{DRIVE}/cinema_oof_probs.npy'}
for name, path in others.items():
    if os.path.exists(path):
        P[name] = np.load(path)

def ece(Pm, yy, bins=15):
    conf, acc = Pm.max(1), (Pm.argmax(1) == yy).astype(float)
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

def metrics(Pm, i):
    return (balanced_accuracy_score(y[i], Pm[i].argmax(1)),
            roc_auc_score(y[i], Pm[i], multi_class='ovr', average='macro', labels=range(len(classes))), ece(Pm[i], y[i]))

rng = np.random.default_rng(SEED)
boots = [i for i in (rng.integers(0, len(y), len(y)) for _ in range(B)) if len(np.unique(y[i])) == len(classes)]
rows = []
for name, Pm in P.items():
    pt, bs = metrics(Pm, np.arange(len(y))), np.array([metrics(Pm, i) for i in boots])
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
    rows.append(dict(model=name, bal_acc=pt[0], bal_lo=lo[0], bal_hi=hi[0], AUC=pt[1], AUC_lo=lo[1], AUC_hi=hi[1],
                     ECE=pt[2], macro_F1=f1_score(y, Pm.argmax(1), average='macro')))
res = pd.DataFrame(rows)
print('\n', res.round(3).to_string(index=False))

ours = 'concept21_LR_pred (ours)'
for ref in ('challenge_RF_pred', 'concept7_LR_pred (previous)'):
    if ref in P:
        d = np.array([np.subtract(metrics(P[ours], i)[:2], metrics(P[ref], i)[:2]) for i in boots])
        print(f'\nPaired difference: {ours} minus {ref}')
        for j, m in enumerate(['bal_acc', 'AUC']):
            print(f'  {m}: {d[:, j].mean():+.3f}  95% CI [{np.percentile(d[:, j], 2.5):+.3f}, {np.percentile(d[:, j], 97.5):+.3f}]')

print('\nConfusion matrix, ours (rows = truth):', [str(c) for c in classes])
print(confusion_matrix(y, P[ours].argmax(1)))
res.to_csv(f'{OUT}/step2b_results.csv', index=False)
pd.DataFrame(coefs).to_csv(f'{OUT}/step2b_coefficients.csv', index=False)
np.save(f'{OUT}/oof_probs_concept21_LR_pred.npy', P[ours])
print('\nSaved to', OUT)


## Part 3 — Comparators (Section 3.4, results in 4.1 and Table 1)
*Do stronger models do better?* The rule from Part 2 is compared, on the same folds, with a cine-CMR foundation model (linear probe and two fine-tuning runs), a joint concept model and an additive model. The random forest was fitted in Part 2.

### Heart centres (CPU)
Used by the heart-centred crop of CineMA fine-tuning run 2.

In [ ]:
import glob, os, numpy as np, pandas as pd
DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
rows = []
for f in sorted(glob.glob(f'{DRIVE}/step3_challenge_baseline/pred_masks/*.npz')):
    m = np.load(f)['ED'] > 0
    cx, cy, cz = np.argwhere(m).mean(0)
    rows.append(dict(patient=os.path.basename(f)[:-4], cx=cx, cy=cy, cz=cz))
df = pd.DataFrame(rows)
assert len(df) == 100, len(df)
df.to_csv(f'{DRIVE}/heart_centers.csv', index=False)
print(df.head(), '\nsaved', len(df), 'centres to', f'{DRIVE}/heart_centers.csv')


### CineMA (GPU)
The foundation-model comparator. First install the CineMA code from its public repository.

In [ ]:
!git clone -q https://github.com/mathpluscode/CineMA.git 2>/dev/null; pip -q install --no-deps -e CineMA; pip -q install omegaconf hydra-core monai timm


Save the CineMA script to a file, so it can be run three times with different settings. Only the public pretrained weights are used.

In [ ]:
%%writefile /content/step_cinema.py
import os, glob, time, copy, zipfile
import numpy as np, pandas as pd, nibabel as nib, torch, torch.nn.functional as F
from scipy.ndimage import zoom
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from omegaconf import OmegaConf
import cinema
from cinema import ConvViT

MODE = os.environ.get('MODE', 'frozen')
KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if KAGGLE:
    DRIVE = '/kaggle/working'
    ROOT = os.path.dirname(os.path.dirname(sorted(glob.glob('/kaggle/input/**/patient001/Info.cfg', recursive=True))[0]))
    FOLDS = sorted(glob.glob('/kaggle/input/**/folds.csv', recursive=True))[0]
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except (ImportError, NotImplementedError):
        pass
    DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
    ROOT = '/content/acdc/training'
    FOLDS = f'{DRIVE}/folds.csv'
    if not os.path.isdir(ROOT):
        zipfile.ZipFile('/content/drive/MyDrive/ACDC_calibration_article/training-20261021T163705Z-1-001.zip').extractall('/content/acdc')
ROI = os.environ.get('ROI', '0') == '1'
OUT = f"{DRIVE}/step_cinema_{MODE}{'_roi' if ROI else ''}{os.environ.get('RUN_TAG', '')}"
os.makedirs(OUT, exist_ok=True)
SEED, PATCH = 42, (192, 192, 16)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'MODE {MODE} | device {dev} | data {ROOT} | out {OUT}')

def read_cfg(p):
    return {k.strip(): v.strip() for k, v in (l.split(':', 1) for l in open(p) if ':' in l)}

def nii(base):
    for e in ('.nii.gz', '.nii'):
        if os.path.exists(base + e):
            return base + e
    raise FileNotFoundError(base)

def fit(v, shape):
    out = np.zeros(shape, np.float32)
    src, dst = [], []
    for a, t in zip(v.shape, shape):
        n = min(a, t)
        src.append(slice((a - n) // 2, (a - n) // 2 + n))
        dst.append(slice((t - n) // 2, (t - n) // 2 + n))
    out[tuple(dst)] = v[tuple(src)]
    return out

CENTERS = None
if ROI:
    cpath = sorted(glob.glob('/kaggle/input/**/heart_centers.csv', recursive=True)) if KAGGLE else [f'{DRIVE}/heart_centers.csv']
    CENTERS = pd.read_csv(cpath[0]).set_index('patient')

def fit_at(v, shape, centre):
    """crop/pad v to shape so that `centre` (voxel coords) lands in the middle."""
    out = np.zeros(shape, np.float32)
    src, dst = [], []
    for a, t, c in zip(v.shape, shape, centre):
        s0 = int(round(c)) - t // 2
        lo, hi = max(s0, 0), min(s0 + t, a)
        src.append(slice(lo, hi)); dst.append(slice(lo - s0, lo - s0 + (hi - lo)))
    out[tuple(dst)] = v[tuple(src)]
    return out

def load(pid):
    d = f'{ROOT}/{pid}'
    info = read_cfg(f'{d}/Info.cfg')
    ch = []
    for fr in (int(info['ED']), int(info['ES'])):
        img = nib.load(nii(f'{d}/{pid}_frame{fr:02d}'))
        sx, sy, sz = img.header.get_zooms()[:3]
        r = zoom(img.get_fdata().astype(np.float32), (sx, sy, sz / 10.0), order=1)
        if ROI:
            c = CENTERS.loc[pid]
            v = fit_at(r, PATCH, (c.cx * sx, c.cy * sy, c.cz * sz / 10.0))
        else:
            v = fit(r, PATCH)
        ch.append((v - v.min()) / (v.max() - v.min() + 1e-8))
    return np.stack(ch)

folds = pd.read_csv(FOLDS)
classes = sorted(folds.label.unique())
y = np.searchsorted(classes, folds.label.values)
f = folds.fold.values
t0 = time.time()
X = np.stack([load(p) for p in folds.patient])
print(f'loaded {X.shape} in {time.time() - t0:.0f}s')

cfg = OmegaConf.load(os.path.join(os.path.dirname(cinema.__file__), 'classification', 'acdc', 'config.yaml'))
cfg.model.views = ['sax']
cfg.model.ckpt_path = None
assert list(cfg.data.pathology) == classes, (list(cfg.data.pathology), classes)

def inner_split(tr_idx):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
    a, b = next(sss.split(tr_idx, y[tr_idx]))
    return tr_idx[a], tr_idx[b]

P = np.zeros((len(y), len(classes)))
curves, picks = [], []

if MODE == 'frozen':
    model = ConvViT.from_pretrained(config=cfg, freeze=True).to(dev).eval()
    feats = []
    with torch.no_grad():
        for i in range(len(X)):
            xd = model.feature_forward(image_dict={'sax': torch.from_numpy(X[i:i + 1]).to(dev)}, mask_dict=None)
            cls_tok = xd['cls'].reshape(1, -1)
            tok = xd['sax'].reshape(1, -1, xd['sax'].shape[-1]).mean(1)
            feats.append(torch.cat([cls_tok, tok], 1).cpu().numpy()[0])
    Fm = np.stack(feats)
    pd.DataFrame(Fm).assign(patient=folds.patient.values).set_index('patient').to_csv(f'{DRIVE}/cinema_features.csv')
    for k in range(5):
        tr, te = f != k, f == k
        grid = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, class_weight='balanced')),
                            {'logisticregression__C': [0.001, 0.01, 0.1, 1, 10]},
                            cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy')
        grid.fit(Fm[tr], y[tr])
        P[te] = grid.predict_proba(Fm[te])
        picks.append(dict(fold=k, C=grid.best_params_['logisticregression__C']))
        print('fold', k, grid.best_params_)

if MODE == 'finetune':
    from cinema.convvit import param_groups_lr_decay
    from cinema.optim import GradScaler, adjust_learning_rate
    amp_dtype = torch.bfloat16 if os.environ.get('AMP', 'fp16') == 'bf16' else torch.float16
    EPOCHS, BATCH = int(os.environ.get('EPOCHS', 100)), int(os.environ.get('BATCH', 4))
    LR = float(os.environ.get('LR', cfg.train.lr * BATCH / 64))
    WARM, EVAL, PATIENCE = 10, 5, int(os.environ.get('PATIENCE', 5))
    print(f'EPOCHS {EPOCHS} | BATCH {BATCH} | LR {LR:.2e} | PATIENCE {PATIENCE} | ROI {ROI} | amp {amp_dtype}')

    def augment(x):
        B = x.size(0)
        if torch.rand(1) < cfg.transform.prob:
            g = torch.empty(B, 1, 1, 1, 1, device=x.device).uniform_(*cfg.transform.gamma)
            x = x.clamp(min=0) ** g
        if torch.rand(1) < cfg.transform.prob:
            x = x + 0.1 * torch.randn_like(x)
        if torch.rand(1) < cfg.transform.prob:
            th = torch.empty(B, device=x.device).uniform_(-np.pi, np.pi)
            s = 1 + torch.empty(B, device=x.device).uniform_(-cfg.transform.scale_range, cfg.transform.scale_range)
            t = torch.empty(B, 2, device=x.device).uniform_(-60, 60) / (PATCH[0] / 2)
            A = torch.stack([torch.stack([torch.cos(th) * s, -torch.sin(th) * s, t[:, 0]], 1),
                             torch.stack([torch.sin(th) * s, torch.cos(th) * s, t[:, 1]], 1)], 1)
            C, H, W, D = x.shape[1:]
            z = x.permute(0, 1, 4, 2, 3).reshape(B, C * D, H, W)
            z = F.grid_sample(z, F.affine_grid(A, z.shape, align_corners=False), align_corners=False)
            x = z.reshape(B, C, D, H, W).permute(0, 1, 3, 4, 2)
        return x

    @torch.no_grad()
    def predict(model, idx):
        model.eval()
        out = []
        for i in idx:
            with torch.autocast('cuda', dtype=amp_dtype, enabled=dev == 'cuda'):
                out.append(model({'sax': torch.from_numpy(X[i:i + 1]).to(dev)}).float().softmax(1).cpu().numpy()[0])
        return np.stack(out)

    done = set()
    if os.path.exists(f'{OUT}/model_selection.csv') and os.path.exists(f'{OUT}/oof_probs_partial.npy'):
        picks = pd.read_csv(f'{OUT}/model_selection.csv').to_dict('records')
        done = {int(r['fold']) for r in picks}
        P = np.load(f'{OUT}/oof_probs_partial.npy')
        if os.path.exists(f'{OUT}/learning_curves.csv'):
            curves = pd.read_csv(f'{OUT}/learning_curves.csv').to_dict('records')
        print('resuming, folds already done:', sorted(done))
    for k in range(5):
        if k in done:
            continue
        t0 = time.time()
        tr, va = inner_split(np.where(f != k)[0])
        te = np.where(f == k)[0]
        torch.manual_seed(SEED + k)
        model = ConvViT.from_pretrained(config=cfg, freeze=False)
        model.set_grad_ckpt(cfg.grad_ckpt)
        model.to(dev)
        opt = torch.optim.AdamW(param_groups_lr_decay(model, no_weight_decay_list=[], weight_decay=cfg.train.weight_decay,
                                                      layer_decay=cfg.train.layer_decay), lr=LR, betas=tuple(cfg.train.betas))
        scaler = GradScaler()
        best, best_ep, bad, best_state = -1.0, 0, 0, None
        n_steps = max(1, len(tr) // BATCH)
        for ep in range(EPOCHS):
            model.train()
            perm = np.random.default_rng(SEED * 100 + k * 1000 + ep).permutation(tr)
            tl = 0.0
            for s in range(n_steps):
                adjust_learning_rate(opt, step=s / n_steps + ep, warmup_steps=WARM, max_n_steps=EPOCHS,
                                     lr=LR, min_lr=cfg.train.min_lr)
                b = perm[s * BATCH:(s + 1) * BATCH]
                xb = augment(torch.from_numpy(X[b]).to(dev))
                yb = torch.from_numpy(y[b]).to(dev)
                with torch.autocast('cuda', dtype=amp_dtype, enabled=dev == 'cuda'):
                    loss = F.cross_entropy(model({'sax': xb}).float(), yb, label_smoothing=cfg.train.label_smoothing)
                scaler(loss, opt, clip_grad=cfg.train.clip_grad, parameters=model.parameters(), update_grad=True)
                opt.zero_grad()
                tl += loss.item()
            row = dict(fold=k, epoch=ep + 1, train_loss=tl / n_steps)
            if (ep + 1) % EVAL == 0:
                pv = predict(model, va)
                auc = roc_auc_score(y[va], pv, multi_class='ovo', average='macro', labels=range(len(classes)))
                row.update(val_auc=auc, val_bal_acc=balanced_accuracy_score(y[va], pv.argmax(1)))
                print(f'  fold {k} epoch {ep + 1}: train loss {tl / n_steps:.3f} | inner-val AUC {auc:.3f}', flush=True)
                if auc > best + 1e-4:
                    best, best_ep, bad = auc, ep + 1, 0
                    best_state = copy.deepcopy(model.state_dict())
                else:
                    bad += 1
                if bad >= PATIENCE:
                    curves.append(row)
                    break
            curves.append(row)
        model.load_state_dict(best_state)
        torch.save({kk: v.half() for kk, v in best_state.items()}, f'{OUT}/cinema_finetune_fold{k}.pt')
        P[te] = predict(model, te)
        picks.append(dict(fold=k, best_epoch=best_ep, inner_val_auc=best))
        print(f'fold {k}: best epoch {best_ep}, inner-val AUC {best:.3f}, stopped at {ep + 1}, {time.time() - t0:.0f}s')
        pd.DataFrame(curves).to_csv(f'{OUT}/learning_curves.csv', index=False)
        pd.DataFrame(picks).to_csv(f'{OUT}/model_selection.csv', index=False)
        np.save(f'{OUT}/oof_probs_partial.npy', P)
        del model, opt
        torch.cuda.empty_cache()

np.save(f'{OUT}/oof_probs.npy', P)
if MODE == 'frozen':
    np.save(f'{DRIVE}/cinema_oof_probs.npy', P)
pd.DataFrame(picks).to_csv(f'{OUT}/model_selection.csv', index=False)
print(f'\nCineMA {MODE}: balanced acc {balanced_accuracy_score(y, P.argmax(1)):.3f} | '
      f'macro AUC (OvR) {roc_auc_score(y, P, multi_class="ovr", average="macro"):.3f}')
print('Saved to', OUT)


**Linear probe.** The encoder stays frozen and a logistic regression is trained on its features. This run also writes `cinema_features.csv`, used later by the additive model.

In [ ]:
os.environ.update(MODE='frozen')
%run /content/step_cinema.py


**Fine-tuning run 1.** Centred crop, the authors' learning rate scaled to batch 4, up to 100 epochs, early stopping after 5 evaluations without improvement.

In [ ]:
os.environ.update(MODE='finetune', ROI='0', EPOCHS='100', PATIENCE='5')
os.environ.pop('LR', None)
%run /content/step_cinema.py


**Fine-tuning run 2.** Crop centred on the heart, learning rate 2.5 × 10⁻⁴, up to 200 epochs, early stopping after 10 evaluations. The settings are cleared at the end.

In [ ]:
os.environ.update(MODE='finetune', ROI='1', EPOCHS='200', PATIENCE='10', LR='2.5e-4')
%run /content/step_cinema.py
for k in ('MODE', 'ROI', 'EPOCHS', 'PATIENCE', 'LR'):
    os.environ.pop(k, None)


### Joint concept model (GPU, ~5–10 min per fold)
Same accuracy as the rule, but its measurements drift from their meaning (Section 4.1).

In [ ]:
import os, glob, time, copy, zipfile
import numpy as np, pandas as pd, nibabel as nib, torch, torch.nn as nn, torch.nn.functional as F
from scipy.ndimage import distance_transform_edt
from torch.utils.checkpoint import checkpoint
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, f1_score, confusion_matrix

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if KAGGLE:
    DRIVE = '/kaggle/working'
    ROOT = os.path.dirname(os.path.dirname(sorted(glob.glob('/kaggle/input/**/patient001/Info.cfg', recursive=True))[0]))
    FOLDS = sorted(glob.glob('/kaggle/input/**/folds.csv', recursive=True))[0]
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except (ImportError, NotImplementedError):
        pass
    DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
    ROOT = '/content/acdc/training'
    FOLDS = f'{DRIVE}/folds.csv'
    if not os.path.isdir(ROOT):
        zipfile.ZipFile('/content/drive/MyDrive/ACDC_calibration_article/training-20261021T163705Z-1-001.zip').extractall('/content/acdc')
STEP1 = os.environ.get('STEP1_DIR', f'{DRIVE}/step1_unet')
OUT = f"{DRIVE}/step4_joint{os.environ.get('RUN_TAG', '')}"
os.makedirs(OUT, exist_ok=True)
LAMBDA_C, LAMBDA_S = float(os.environ.get('LAMBDA_C', 1.0)), float(os.environ.get('LAMBDA_S', 1.0))
EPOCHS, BP, SEED, SIZE, WIDTH, BOOT = int(os.environ.get('EPOCHS', 30)), 4, 42, 224, 32, 2000
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device {dev} | LAMBDA_C {LAMBDA_C} | LAMBDA_S {LAMBDA_S} | EPOCHS {EPOCHS} | out {OUT}')

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(s, c=32, n=4, p=0.1):
        super().__init__()
        s.e1, s.e2, s.e3, s.e4 = block(1, c), block(c, 2 * c), block(2 * c, 4 * c), block(4 * c, 8 * c)
        s.b = block(8 * c, 16 * c)
        s.drop = nn.Dropout2d(p)
        s.u4, s.d4 = nn.ConvTranspose2d(16 * c, 8 * c, 2, 2), block(16 * c, 8 * c)
        s.u3, s.d3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, 2), block(8 * c, 4 * c)
        s.u2, s.d2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, 2), block(4 * c, 2 * c)
        s.u1, s.d1 = nn.ConvTranspose2d(2 * c, c, 2, 2), block(2 * c, c)
        s.out = nn.Conv2d(c, n, 1)

    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(F.max_pool2d(e1, 2)); e3 = s.e3(F.max_pool2d(e2, 2)); e4 = s.e4(F.max_pool2d(e3, 2))
        b = s.drop(s.b(F.max_pool2d(e4, 2)))
        d4 = s.d4(torch.cat([s.u4(b), e4], 1)); d3 = s.d3(torch.cat([s.u3(d4), e3], 1))
        d2 = s.d2(torch.cat([s.u2(d3), e2], 1)); d1 = s.d1(torch.cat([s.u1(d2), e1], 1))
        return s.out(d1)

def read_cfg(p):
    return {k.strip(): v.strip() for k, v in (l.split(':', 1) for l in open(p) if ':' in l)}

def nii(base):
    for e in ('.nii.gz', '.nii'):
        if os.path.exists(base + e):
            return base + e
    raise FileNotFoundError(base)

def norm(v):
    lo, hi = np.percentile(v, [1, 99])
    v = np.clip(v, lo, hi)
    return ((v - v.mean()) / (v.std() + 1e-8)).astype(np.float32)

def rs(a, mode):
    t = torch.from_numpy(np.ascontiguousarray(a))[:, None].float()
    if mode == 'nearest':
        return F.interpolate(t, size=(SIZE, SIZE), mode='nearest')[:, 0]
    return F.interpolate(t, size=(SIZE, SIZE), mode='bilinear', align_corners=False)[:, 0]

folds = pd.read_csv(FOLDS)
order = folds.patient.values
classes = np.array(sorted(folds.label.unique()))
y = np.searchsorted(classes, folds.label.values)
f = folds.fold.values
D = []
t0 = time.time()
for p in order:
    d, info = f'{ROOT}/{p}', read_cfg(f'{ROOT}/{p}/Info.cfg')
    item = {}
    for ph in ('ED', 'ES'):
        img = nib.load(nii(f'{d}/{p}_frame{int(info[ph]):02d}'))
        gt = np.rint(nib.load(nii(f'{d}/{p}_frame{int(info[ph]):02d}_gt')).get_fdata()).astype(np.int64)
        v = norm(img.get_fdata())
        item[ph] = dict(x=rs(v.transpose(2, 0, 1), 'bilinear')[:, None], ys=rs(gt.transpose(2, 0, 1), 'nearest').long(),
                        gt=gt, hw=v.shape[:2], vox=float(np.prod(img.header.get_zooms()[:3])) / 1000,
                        zoom=tuple(float(z) for z in img.header.get_zooms()[:3]))
    D.append(item)
print(f'loaded {len(D)} patients in {time.time() - t0:.0f}s')

def wt_stats(myo, zooms):
    t = [2 * distance_transform_edt(myo[:, :, z], sampling=zooms[:2]).max() - np.mean(zooms[:2])
         for z in range(myo.shape[2]) if myo[:, :, z].sum() >= 10]
    t = np.array(t) if t else np.array([0.0])
    return [t.max(), t.mean(), t.std(), t.min()]

def concepts(prob, item):
    """prob[ph]: (Z, 4, H, W) torch probabilities (or one-hot). Returns (21,) torch tensor."""
    v = {f'{s}_{ph}': prob[ph][:, c].sum() * item[ph]['vox'] for ph in ('ED', 'ES') for s, c in (('RV', 1), ('MYO', 2), ('LV', 3))}
    e = 1e-6
    diff = [v['RV_ED'], v['MYO_ED'], v['LV_ED'], v['RV_ES'], v['MYO_ES'], v['LV_ES'],
            100 * (v['LV_ED'] - v['LV_ES']) / (v['LV_ED'] + e), 100 * (v['RV_ED'] - v['RV_ES']) / (v['RV_ED'] + e),
            1.05 * v['MYO_ED'], v['RV_ED'] / (v['LV_ED'] + e), v['RV_ES'] / (v['LV_ES'] + e),
            v['MYO_ED'] / (v['LV_ED'] + e), v['MYO_ES'] / (v['LV_ES'] + e)]
    wt = []
    for ph in ('ED', 'ES'):
        hard = (prob[ph].detach().argmax(1) == 2).permute(1, 2, 0).cpu().numpy()
        wt += wt_stats(hard, item[ph]['zoom'])
    return torch.stack(diff + [torch.tensor(w, dtype=torch.float32, device=diff[0].device) for w in wt])

NAMES = ['RV_ED', 'MYO_ED', 'LV_ED', 'RV_ES', 'MYO_ES', 'LV_ES', 'LV_EF', 'RV_EF', 'LV_mass', 'RV_LV_ED', 'RV_LV_ES',
         'MYO_LV_ED', 'MYO_LV_ES', 'WT_ED_max', 'WT_ED_mean', 'WT_ED_std', 'WT_ED_min',
         'WT_ES_max', 'WT_ES_mean', 'WT_ES_std', 'WT_ES_min']
N_DIFF = 13

def onehot_prob(item, ph):
    return F.one_hot(torch.from_numpy(item[ph]['gt'].transpose(2, 0, 1)), 4).permute(0, 3, 1, 2).float()

C_ref = torch.stack([concepts({ph: onehot_prob(it, ph) for ph in ('ED', 'ES')}, it) for it in D]).numpy()

def forward_probs(net, item, ph, aug_k=None):
    x = item[ph]['x'].to(dev)
    if aug_k is not None:
        x = torch.rot90(x, aug_k[0], (-2, -1)).flip(-1) if aug_k[1] else torch.rot90(x, aug_k[0], (-2, -1))
    if torch.is_grad_enabled():
        logits = torch.cat([checkpoint(net, x[j:j + 4], use_reentrant=False)
                            for j in range(0, x.shape[0], 4)]).float()
    else:
        logits = net(x).float()
    if aug_k is not None:
        logits = logits.flip(-1) if aug_k[1] else logits
        logits = torch.rot90(logits, -aug_k[0], (-2, -1))
    p = F.interpolate(logits.softmax(1), size=item[ph]['hw'], mode='bilinear', align_corners=False)
    return logits, p

def seg_loss(logits, ys):
    p = logits.softmax(1)
    oh = F.one_hot(ys, 4).permute(0, 3, 1, 2).float()
    dice = 1 - ((2 * (p * oh).sum((0, 2, 3)) + 1) / (p.sum((0, 2, 3)) + oh.sum((0, 2, 3)) + 1))[1:].mean()
    return F.cross_entropy(logits, ys) + dice

P_joint = np.zeros((len(y), len(classes)))
C_joint = np.zeros_like(C_ref)
curves, picks, dice_rows = [], [], []
for k in range(5):
    t0 = time.time()
    tr_all = np.where(f != k)[0]
    a, b = next(StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED).split(tr_all, y[tr_all]))
    tr, va, te = tr_all[a], tr_all[b], np.where(f == k)[0]

    grid = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced')),
                        {'logisticregression__C': [0.01, 0.1, 1, 10, 100]},
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy')
    grid.fit(C_ref[tr_all], y[tr_all])
    sc, lr = grid.best_estimator_.named_steps['standardscaler'], grid.best_estimator_.named_steps['logisticregression']
    mu, sd = torch.tensor(sc.mean_, dtype=torch.float32, device=dev), torch.tensor(sc.scale_, dtype=torch.float32, device=dev)
    head = nn.Linear(21, len(classes)).to(dev)
    with torch.no_grad():
        head.weight.copy_(torch.tensor(lr.coef_, dtype=torch.float32)); head.bias.copy_(torch.tensor(lr.intercept_, dtype=torch.float32))
    cw = torch.tensor(len(tr) / (len(classes) * np.bincount(y[tr], minlength=len(classes))), dtype=torch.float32, device=dev)

    np.random.seed(SEED + k); torch.manual_seed(SEED + k)
    net = UNet(c=WIDTH).to(dev)
    net.load_state_dict(torch.load(f'{STEP1}/unet_fold{k}.pt', map_location=dev))
    opt = torch.optim.AdamW([{'params': net.parameters(), 'lr': 1e-4}, {'params': head.parameters(), 'lr': 1e-3}], weight_decay=1e-4)

    def run(idx, train):
        net.train(train); head.train(train)
        if train:
            for m in net.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()
        logits_all, conc_all = [], []
        with torch.set_grad_enabled(train):
            for i in idx:
                augk = (int(np.random.randint(4)), bool(np.random.randint(2))) if train else None
                lp = {ph: forward_probs(net, D[i], ph, augk) for ph in ('ED', 'ES')}
                c = concepts({ph: lp[ph][1].float() for ph in ('ED', 'ES')}, D[i])
                if train:
                    yield i, lp, c
                else:
                    logits_all.append(head((c - mu) / sd)); conc_all.append(c)
        if not train:
            yield torch.stack(logits_all), torch.stack(conc_all)

    def evaluate(idx):
        lo, cc = next(run(idx, False))
        return lo, cc

    best, best_state, best_ep = np.inf, None, 0
    lo0, _ = evaluate(va)
    best = F.cross_entropy(lo0, torch.tensor(y[va], device=dev)).item()
    best_state = (copy.deepcopy(net.state_dict()), copy.deepcopy(head.state_dict()))
    curves.append(dict(fold=k, epoch=0, val_nll=best, val_bal_acc=balanced_accuracy_score(y[va], lo0.argmax(1).cpu().numpy())))
    for ep in range(1, EPOCHS + 1):
        rng = np.random.default_rng(SEED * 100 + k * 1000 + ep)
        perm = rng.permutation(tr)
        tot = {'dx': 0.0, 'concept': 0.0, 'seg': 0.0}
        for s in range(0, len(perm) - BP + 1, BP):
            opt.zero_grad(set_to_none=True)
            for i, lp, c in run(perm[s:s + BP], True):
                z = (c - mu) / sd
                l_dx = cw[y[i]] * F.cross_entropy(head(z)[None], torch.tensor([y[i]], device=dev))
                zr = (torch.tensor(C_ref[i], device=dev) - mu) / sd
                l_c = F.smooth_l1_loss(z[:N_DIFF], zr[:N_DIFF])
                l_s = sum(seg_loss(lp[ph][0], D[i][ph]['ys'].to(dev)) for ph in ('ED', 'ES')) / 2
                ((l_dx + LAMBDA_C * l_c + LAMBDA_S * l_s) / BP).backward()
                for key, val in (('dx', l_dx), ('concept', l_c), ('seg', l_s)):
                    tot[key] += val.item() / BP
                del lp, c, z
            torch.nn.utils.clip_grad_norm_(list(net.parameters()) + list(head.parameters()), 5.0)
            opt.step()
        lo, _ = evaluate(va)
        nll = F.cross_entropy(lo, torch.tensor(y[va], device=dev)).item()
        n_b = max(1, len(perm) // BP)
        curves.append(dict(fold=k, epoch=ep, train_dx=tot['dx'] / n_b, train_concept=tot['concept'] / n_b,
                           train_seg=tot['seg'] / n_b, val_nll=nll,
                           val_bal_acc=balanced_accuracy_score(y[va], lo.argmax(1).cpu().numpy())))
        if nll < best:
            best, best_ep = nll, ep
            best_state = (copy.deepcopy(net.state_dict()), copy.deepcopy(head.state_dict()))

    net.load_state_dict(best_state[0]); head.load_state_dict(best_state[1])
    torch.save({'unet': best_state[0], 'head': best_state[1], 'mu': mu.cpu(), 'sd': sd.cpu()}, f'{OUT}/joint_fold{k}.pt')
    lo, cc = evaluate(te)
    P_joint[te] = lo.softmax(1).detach().cpu().numpy()
    C_joint[te] = cc.detach().cpu().numpy()
    picks.append(dict(fold=k, best_epoch=best_ep, inner_val_nll=best, C=grid.best_params_['logisticregression__C']))
    print(f'fold {k}: best epoch {best_ep}/{EPOCHS} (0 = sequential), inner-val NLL {best:.3f}, {time.time() - t0:.0f}s')
    pd.DataFrame(curves).to_csv(f'{OUT}/learning_curves.csv', index=False)
    with torch.no_grad():
        for i in te:
            row = dict(patient=order[i], fold=k)
            for ph in ('ED', 'ES'):
                pred = forward_probs(net, D[i], ph)[1].argmax(1).cpu().numpy().transpose(1, 2, 0)
                for n, c in (('RV', 1), ('MYO', 2), ('LV', 3)):
                    a_, b_ = pred == c, D[i][ph]['gt'] == c
                    row[f'{ph}_{n}'] = 2 * (a_ & b_).sum() / max(a_.sum() + b_.sum(), 1)
            dice_rows.append(row)
    del net, head, opt
    torch.cuda.empty_cache()

np.save(f'{OUT}/oof_probs_joint.npy', P_joint)
pd.DataFrame(C_joint, columns=NAMES).assign(patient=order, source='joint').to_csv(f'{OUT}/concepts_joint.csv', index=False)
pd.DataFrame(picks).to_csv(f'{OUT}/model_selection.csv', index=False)
dice = pd.DataFrame(dice_rows)
dice.to_csv(f'{OUT}/dice_test.csv', index=False)
print('\nHeld-out 3D Dice, joint U-Net:\n', dice.drop(columns=['patient', 'fold']).mean().round(3).to_string())

P = {'joint (ours)': P_joint}
for name, path in (('sequential concept21 (step 2b)', f'{DRIVE}/step2b_concepts21/oof_probs_concept21_LR_pred.npy'),
                   ('challenge_RF_pred (step 3)', f'{DRIVE}/step3_challenge_baseline/oof_probs_challenge_RF_pred.npy')):
    if os.path.exists(path):
        P[name] = np.load(path)

def ece(Pm, yy, bins=15):
    conf, acc = Pm.max(1), (Pm.argmax(1) == yy).astype(float)
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo_, hi_ in zip(edges[:-1], edges[1:]):
        m = (conf > lo_) & (conf <= hi_)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

def metrics(Pm, i):
    return (balanced_accuracy_score(y[i], Pm[i].argmax(1)),
            roc_auc_score(y[i], Pm[i], multi_class='ovr', average='macro', labels=range(len(classes))), ece(Pm[i], y[i]))

rng = np.random.default_rng(SEED)
boots = [i for i in (rng.integers(0, len(y), len(y)) for _ in range(BOOT)) if len(np.unique(y[i])) == len(classes)]
rows = []
for name, Pm in P.items():
    pt, bs = metrics(Pm, np.arange(len(y))), np.array([metrics(Pm, i) for i in boots])
    lo_, hi_ = np.percentile(bs, [2.5, 97.5], axis=0)
    rows.append(dict(model=name, bal_acc=pt[0], bal_lo=lo_[0], bal_hi=hi_[0], AUC=pt[1], AUC_lo=lo_[1], AUC_hi=hi_[1],
                     ECE=pt[2], macro_F1=f1_score(y, Pm.argmax(1), average='macro')))
res = pd.DataFrame(rows)
print('\n', res.round(3).to_string(index=False))
for ref in list(P)[1:]:
    d = np.array([np.subtract(metrics(P['joint (ours)'], i)[:2], metrics(P[ref], i)[:2]) for i in boots])
    print(f'\nPaired difference: joint minus {ref}')
    for j, m in enumerate(['bal_acc', 'AUC']):
        print(f'  {m}: {d[:, j].mean():+.3f}  95% CI [{np.percentile(d[:, j], 2.5):+.3f}, {np.percentile(d[:, j], 97.5):+.3f}]')

print('\nConcept agreement on held-out patients (joint minus reference): bias, 95% LoA, SD ratio')
ba = pd.DataFrame([dict(concept=n, bias=(C_joint[:, j] - C_ref[:, j]).mean(),
                        loa=1.96 * (C_joint[:, j] - C_ref[:, j]).std(ddof=1),
                        sd_ratio=C_joint[:, j].std() / C_ref[:, j].std()) for j, n in enumerate(NAMES)])
print(ba.round(2).to_string(index=False))
ba.to_csv(f'{OUT}/concept_agreement.csv', index=False)
res.to_csv(f'{OUT}/step4_results.csv', index=False)
print('\nConfusion matrix, joint (rows = truth):', [str(c) for c in classes])
print(confusion_matrix(y, P_joint.argmax(1)))
print('\nSaved to', OUT)


### Additive model (CPU)
Rule logits plus a penalised CineMA residual; completes Table 1.

In [ ]:
import os, numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy.special import softmax, log_softmax
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, f1_score, confusion_matrix

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
CINEMA_FEATS = os.environ.get('CINEMA_FEATS', f'{DRIVE}/cinema_features.csv')
OUT = f'{DRIVE}/step5_c1_additive'
os.makedirs(OUT, exist_ok=True)
SEED, B, NPC = 42, 2000, 32
LAMBDAS = [np.inf, 100.0, 30.0, 10.0, 3.0, 1.0]

folds = pd.read_csv(f'{DRIVE}/folds.csv')
order = folds.patient.values
classes = np.array(sorted(folds.label.unique()))
y = np.searchsorted(classes, folds.label.values)
f = folds.fold.values
n, K = len(y), len(classes)
rf = pd.read_csv(f'{DRIVE}/step3_challenge_baseline/rich_features.csv')
FEATS = [c for c in rf.columns if c not in ('patient', 'source')]
C_gt = rf[rf.source == 'gt'].set_index('patient').loc[order, FEATS].values
C_pr = rf[rf.source == 'pred'].set_index('patient').loc[order, FEATS].values
R_all = pd.read_csv(CINEMA_FEATS).set_index('patient').loc[order].values
print(f'{n} patients | {len(FEATS)} concepts | {R_all.shape[1]} transformer features')

def concept_model():
    return GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced')),
                        {'logisticregression__C': [0.01, 0.1, 1, 10, 100]},
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy')

def fit_residual(Z, Lc, yy, lam, w):
    """multinomial residual V (d x K) on top of fixed concept logits Lc; class weights w."""
    if np.isinf(lam):
        return np.zeros((Z.shape[1], K))
    Y = np.eye(K)[yy]
    sw = w[yy] / w[yy].sum()
    def obj(v):
        V = v.reshape(Z.shape[1], K)
        L = Lc + Z @ V
        loss = -(sw * (Y * log_softmax(L, 1)).sum(1)).sum() + lam / len(yy) * (V ** 2).sum()
        G = Z.T @ ((softmax(L, 1) - Y) * sw[:, None]) + 2 * lam / len(yy) * V
        return loss, G.ravel()
    r = minimize(obj, np.zeros(Z.shape[1] * K), jac=True, method='L-BFGS-B', options={'maxiter': 2000})
    return r.x.reshape(Z.shape[1], K)

def crossfit_concept_logits(idx):
    """concept logits for training patients, each predicted by a model that did not see it."""
    Lc = np.zeros((len(idx), K))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(idx, y[idx]):
        m = concept_model().fit(C_gt[idx[a]], y[idx[a]])
        Lc[b] = m.decision_function(C_gt[idx[b]])
    return Lc

P = np.zeros((n, K)); Lc_te_all = np.zeros((n, K)); Lr_te_all = np.zeros((n, K)); picks = []
for k in range(5):
    tr, te = np.where(f != k)[0], np.where(f == k)[0]
    w = len(tr) / (K * np.bincount(y[tr], minlength=K))
    prep = make_pipeline(StandardScaler(), PCA(n_components=NPC, random_state=SEED)).fit(R_all[tr])
    Z_tr, Z_te = prep.transform(R_all[tr]), prep.transform(R_all[te])
    Z_tr, Z_te = Z_tr / Z_tr.std(0), Z_te / Z_tr.std(0)
    Lc_tr = crossfit_concept_logits(tr)

    scores = {}
    for lam in LAMBDAS:
        s = []
        for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, y[tr]):
            V = fit_residual(Z_tr[a], Lc_tr[a], y[tr][a], lam, w)
            s.append(-np.mean(log_softmax(Lc_tr[b] + Z_tr[b] @ V, 1)[np.arange(len(b)), y[tr][b]]))
        scores[lam] = np.mean(s)
    lam = min(scores, key=scores.get)

    cm = concept_model().fit(C_gt[tr], y[tr])
    V = fit_residual(Z_tr, Lc_tr, y[tr], lam, w)
    Lc_te = cm.decision_function(C_pr[te])
    Lr_te = Z_te @ V
    P[te] = softmax(Lc_te + Lr_te, 1)
    Lc_te_all[te], Lr_te_all[te] = Lc_te, Lr_te
    picks.append(dict(fold=k, lam=lam, **{f'nll_lam_{l}': v for l, v in scores.items()}))
    print(f'fold {k}: lambda = {lam} (inner NLL ' + ', '.join(f'{l}: {v:.3f}' for l, v in scores.items()) + ')')

pred = P.argmax(1)
c_part = Lc_te_all - Lc_te_all.mean(1, keepdims=True)
r_part = Lr_te_all - Lr_te_all.mean(1, keepdims=True)
cs, rs = np.abs(c_part[np.arange(n), pred]), np.abs(r_part[np.arange(n), pred])
share = cs / np.maximum(cs + rs, 1e-12)
ev = pd.DataFrame(dict(patient=order, label=folds.label.values, pred=classes[pred], concept_share=share))
print('\nConcept share of the evidence for the predicted class (1.0 = measurements only):')
print(ev.groupby('label').concept_share.describe()[['mean', 'min', '50%']].round(2).to_string())

Pd = {'additive C1 (ours)': P}
for name, path in (('concepts only, 21 (step 2b)', f'{DRIVE}/step2b_concepts21/oof_probs_concept21_LR_pred.npy'),
                   ('challenge RF (step 3)', f'{DRIVE}/step3_challenge_baseline/oof_probs_challenge_RF_pred.npy'),
                   ('CineMA frozen probe', f'{DRIVE}/cinema_oof_probs.npy')):
    if os.path.exists(path):
        Pd[name] = np.load(path)

def ece(Pm, yy, bins=15):
    conf, acc = Pm.max(1), (Pm.argmax(1) == yy).astype(float)
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

def metrics(Pm, i):
    return (balanced_accuracy_score(y[i], Pm[i].argmax(1)),
            roc_auc_score(y[i], Pm[i], multi_class='ovr', average='macro', labels=range(K)), ece(Pm[i], y[i]))

rng = np.random.default_rng(SEED)
boots = [i for i in (rng.integers(0, n, n) for _ in range(B)) if len(np.unique(y[i])) == K]
rows = []
for name, Pm in Pd.items():
    pt, bs = metrics(Pm, np.arange(n)), np.array([metrics(Pm, i) for i in boots])
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
    rows.append(dict(model=name, bal_acc=pt[0], bal_lo=lo[0], bal_hi=hi[0], AUC=pt[1], AUC_lo=lo[1], AUC_hi=hi[1],
                     ECE=pt[2], macro_F1=f1_score(y, Pm.argmax(1), average='macro')))
res = pd.DataFrame(rows)
print('\n', res.round(3).to_string(index=False))
for ref in list(Pd)[1:3]:
    d = np.array([np.subtract(metrics(Pd['additive C1 (ours)'], i), metrics(Pd[ref], i)) for i in boots])
    print(f'\nPaired difference: additive C1 minus {ref}')
    for j, m in enumerate(['bal_acc', 'AUC', 'ECE']):
        print(f'  {m}: {d[:, j].mean():+.3f}  95% CI [{np.percentile(d[:, j], 2.5):+.3f}, {np.percentile(d[:, j], 97.5):+.3f}]')
print('\nConfusion matrix (rows = truth):', [str(c) for c in classes])
print(confusion_matrix(y, pred))

res.to_csv(f'{OUT}/step5_results.csv', index=False)
ev.to_csv(f'{OUT}/evidence_share.csv', index=False)
pd.DataFrame(picks).to_csv(f'{OUT}/model_selection.csv', index=False)
np.save(f'{OUT}/oof_probs_additive.npy', P)
print('\nSaved to', OUT)


## Part 4 — Per-patient uncertainty and the flip flag (Sections 3.5–3.7, results in 4.2)
*How far could each measurement move, and which one could change the diagnosis?* Twenty stochastic segmentations per patient give each measurement its own spread, inflated by one factor κ fitted on training patients; the flip flag perturbs one measurement at a time.

### Stochastic segmentations, κ, flip flag, conformal sets (GPU then CPU)
Table 2 and the flag results of Section 4.2.

In [ ]:
import os, glob, zipfile, time
import numpy as np, pandas as pd, nibabel as nib, torch, torch.nn as nn, torch.nn.functional as F
from scipy.ndimage import distance_transform_edt
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
STEP1 = os.environ.get('STEP1_DIR', f'{DRIVE}/step1_unet')
ROOT = os.environ.get('ACDC_ROOT', '/content/acdc/training')
OUT = f'{DRIVE}/step8_patient_uncertainty'
os.makedirs(OUT, exist_ok=True)
SEED, SIZE, WIDTH = 42, int(os.environ.get('SIZE', 224)), int(os.environ.get('WIDTH', 32))
S, ALPHA = int(os.environ.get('S', 20)), 0.10
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED); rng = np.random.default_rng(SEED)

folds = pd.read_csv(f'{DRIVE}/folds.csv')
order = folds.patient.values
classes = np.array(sorted(folds.label.unique()))
y = np.searchsorted(classes, folds.label.values)
f = folds.fold.values
n, K = len(y), len(classes)
rf = pd.read_csv(f'{DRIVE}/step3_challenge_baseline/rich_features.csv')
FEATS = [c for c in rf.columns if c not in ('patient', 'source')]
C_gt = rf[rf.source == 'gt'].set_index('patient').loc[order, FEATS].values
C_pr = rf[rf.source == 'pred'].set_index('patient').loc[order, FEATS].values

SAMPLES = f'{OUT}/concept_samples.npz'
if not os.path.exists(SAMPLES):
    if not os.path.isdir(ROOT):
        zipfile.ZipFile('/content/drive/MyDrive/ACDC_calibration_article/training-20261021T163705Z-1-001.zip').extractall('/content/acdc')

    def block(i, o):
        return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                             nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

    class UNet(nn.Module):
        def __init__(s, c=32, n=4, p=0.1):
            super().__init__()
            s.e1, s.e2, s.e3, s.e4 = block(1, c), block(c, 2 * c), block(2 * c, 4 * c), block(4 * c, 8 * c)
            s.b = block(8 * c, 16 * c)
            s.drop = nn.Dropout2d(p)
            s.u4, s.d4 = nn.ConvTranspose2d(16 * c, 8 * c, 2, 2), block(16 * c, 8 * c)
            s.u3, s.d3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, 2), block(8 * c, 4 * c)
            s.u2, s.d2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, 2), block(4 * c, 2 * c)
            s.u1, s.d1 = nn.ConvTranspose2d(2 * c, c, 2, 2), block(2 * c, c)
            s.out = nn.Conv2d(c, n, 1)

        def forward(s, x):
            e1 = s.e1(x); e2 = s.e2(F.max_pool2d(e1, 2)); e3 = s.e3(F.max_pool2d(e2, 2)); e4 = s.e4(F.max_pool2d(e3, 2))
            b = s.drop(s.b(F.max_pool2d(e4, 2)))
            d4 = s.d4(torch.cat([s.u4(b), e4], 1)); d3 = s.d3(torch.cat([s.u3(d4), e3], 1))
            d2 = s.d2(torch.cat([s.u2(d3), e2], 1)); d1 = s.d1(torch.cat([s.u1(d2), e1], 1))
            return s.out(d1)

    def read_cfg(p):
        return {k.strip(): v.strip() for k, v in (l.split(':', 1) for l in open(p) if ':' in l)}

    def nii(base):
        for e in ('.nii.gz', '.nii'):
            if os.path.exists(base + e):
                return base + e
        raise FileNotFoundError(base)

    def norm(v):
        lo, hi = np.percentile(v, [1, 99])
        v = np.clip(v, lo, hi)
        return ((v - v.mean()) / (v.std() + 1e-8)).astype(np.float32)

    @torch.no_grad()
    def sample_mask(net, v, s):
        """s = 0: deterministic; s > 0: MC dropout + random flips + mild gamma."""
        H, W, Z = v.shape
        x = torch.from_numpy(np.ascontiguousarray(v.transpose(2, 0, 1)))[:, None].to(dev)
        x = F.interpolate(x, size=(SIZE, SIZE), mode='bilinear', align_corners=False)
        fl = []
        if s > 0:
            g = float(np.exp(rng.normal(0, 0.1)))
            x = torch.sign(x) * x.abs() ** g
            fl = [d for d in (2, 3) if rng.random() < 0.5]
            if fl: x = torch.flip(x, fl)
        p = net(x).float().softmax(1)
        if fl: p = torch.flip(p, fl)
        p = F.interpolate(p, size=(H, W), mode='bilinear', align_corners=False)
        return p.argmax(1).permute(1, 2, 0).cpu().numpy().astype(np.uint8)

    def thickness_per_slice(myo, zooms):
        out = []
        for z in range(myo.shape[2]):
            m = myo[:, :, z]
            if m.sum() >= 10:
                out.append(2 * distance_transform_edt(m, sampling=zooms[:2]).max() - np.mean(zooms[:2]))
        return np.array(out) if out else np.array([0.0])

    def rich_features(lab_ed, lab_es, z_ed, z_es):
        ved, ves = np.prod(z_ed) / 1000, np.prod(z_es) / 1000
        v = {f'{s}_{ph}': (lab == c).sum() * vv for ph, lab, vv in (('ED', lab_ed, ved), ('ES', lab_es, ves))
             for s, c in (('RV', 1), ('MYO', 2), ('LV', 3))}
        t_ed, t_es = thickness_per_slice(lab_ed == 2, z_ed), thickness_per_slice(lab_es == 2, z_es)
        eps = 1e-6
        return dict(**v,
                    LV_EF=100 * (v['LV_ED'] - v['LV_ES']) / max(v['LV_ED'], eps),
                    RV_EF=100 * (v['RV_ED'] - v['RV_ES']) / max(v['RV_ED'], eps),
                    LV_mass=1.05 * v['MYO_ED'],
                    RV_LV_ED=v['RV_ED'] / max(v['LV_ED'], eps), RV_LV_ES=v['RV_ES'] / max(v['LV_ES'], eps),
                    MYO_LV_ED=v['MYO_ED'] / max(v['LV_ED'], eps), MYO_LV_ES=v['MYO_ES'] / max(v['LV_ES'], eps),
                    WT_ED_max=t_ed.max(), WT_ED_mean=t_ed.mean(), WT_ED_std=t_ed.std(), WT_ED_min=t_ed.min(),
                    WT_ES_max=t_es.max(), WT_ES_mean=t_es.mean(), WT_ES_std=t_es.std(), WT_ES_min=t_es.min())

    CS = np.zeros((n, S, len(FEATS)))
    t0 = time.time()
    pos = {p: i for i, p in enumerate(order)}
    for k in range(5):
        net = UNet(c=WIDTH).to(dev)
        net.load_state_dict(torch.load(f'{STEP1}/unet_fold{k}.pt', map_location=dev))
        net.eval(); net.drop.train()
        for p in order[f == k]:
            d = f'{ROOT}/{p}'; info = read_cfg(f'{d}/Info.cfg')
            vols, zs = [], []
            for ph in ('ED', 'ES'):
                img = nib.load(nii(f'{d}/{p}_frame{int(info[ph]):02d}'))
                vols.append(norm(img.get_fdata())); zs.append(tuple(float(z) for z in img.header.get_zooms()[:3]))
            for s in range(S):
                ft = rich_features(sample_mask(net, vols[0], s + 1), sample_mask(net, vols[1], s + 1), zs[0], zs[1])
                CS[pos[p], s] = [ft[c] for c in FEATS]
        print(f'fold {k}: sampled ({time.time() - t0:.0f}s)')
    np.savez_compressed(SAMPLES, CS=CS, feats=np.array(FEATS), patients=order)
CS = np.load(SAMPLES)['CS']
print(f'concept samples: {CS.shape}')

dev_s = CS - CS.mean(1, keepdims=True)
sd = CS.std(1) + 1e-9

def fit_head(idx):
    g = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced')),
                     {'logisticregression__C': [0.01, 0.1, 1, 10, 100]},
                     cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy')
    g.fit(C_gt[idx], y[idx])
    return g.best_estimator_

def fit_kappa(idx):
    """one global inflation factor: median |error| / median sample SD, in SD units (robust)."""
    r = np.abs(C_pr[idx] - C_gt[idx]) / sd[idx]
    return float(np.median(r) / 0.6745)

def predict(model, idx, mode, kappa):
    if mode == 'plain':
        return model.predict_proba(C_pr[idx])
    kap = 1.0 if mode == 'pp_raw' else kappa
    X = C_pr[idx][:, None, :] + kap * dev_s[idx]
    m, s_, D = X.shape
    return model.predict_proba(X.reshape(-1, D)).reshape(m, s_, K).mean(1)

def qhat(s):
    m = len(s); return np.quantile(s, min(1, np.ceil((m + 1) * (1 - ALPHA)) / m), method='higher')

MODES = ['plain', 'pp_raw', 'pp_cal']
P = {m: np.zeros((n, K)) for m in MODES}
SET = {m: np.zeros((n, K), bool) for m in MODES}
kap_log = []
for k in range(5):
    tr, te = np.where(f != k)[0], np.where(f == k)[0]
    cal = {m: np.zeros(len(tr)) for m in MODES}
    for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, y[tr]):
        mdl, kap = fit_head(tr[a]), fit_kappa(tr[a])
        for m in MODES:
            Pc = predict(mdl, tr[b], m, kap)
            cal[m][b] = 1 - Pc[np.arange(len(b)), y[tr[b]]]
    mdl, kap = fit_head(tr), fit_kappa(tr)
    kap_log.append(kap)
    for m in MODES:
        Pt = predict(mdl, te, m, kap)
        P[m][te] = Pt
        SET[m][te] = (1 - Pt) <= qhat(cal[m])
    print(f'fold {k}: kappa = {kap:.2f}')

def ece(Pm, bins=15):
    conf, acc = Pm.max(1), (Pm.argmax(1) == y).astype(float)
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

rows = []
for m in MODES:
    wrong = P[m].argmax(1) != y
    cov = SET[m][np.arange(n), y]
    rows.append(dict(model=m, bal_acc=balanced_accuracy_score(y, P[m].argmax(1)),
                     AUC=roc_auc_score(y, P[m], multi_class='ovr'), ECE=ece(P[m]),
                     NLL=-np.mean(np.log(np.clip(P[m][np.arange(n), y], 1e-12, 1))),
                     err_detect_AUROC=roc_auc_score(wrong, 1 - P[m].max(1)) if 0 < wrong.sum() < n else np.nan,
                     coverage=cov.mean(), mean_size=SET[m].sum(1).mean(), singleton=(SET[m].sum(1) == 1).mean(),
                     **{f'cov_{c}': cov[y == i].mean() for i, c in enumerate(classes)}))
res = pd.DataFrame(rows)
pd.set_option('display.width', 220)
print(f'\nkappa per fold: {np.round(kap_log, 2)} | target coverage {1 - ALPHA:.2f}')
print(res.round(3).to_string(index=False))

from scipy.stats import spearmanr
print('\nSpearman: per-patient sample SD vs real |pred - gt| (per measurement)')
for j, c in enumerate(FEATS):
    if c in ('LV_EF', 'RV_EF', 'LV_ED', 'RV_ED', 'MYO_ED', 'RV_LV_ED', 'MYO_LV_ED', 'WT_ED_max'):
        r, pv = spearmanr(sd[:, j], np.abs(C_pr[:, j] - C_gt[:, j]))
        print(f'  {c:10s}: rho = {r:+.2f}  p = {pv:.2g}')

expl = []
for k in range(5):
    tr, te = np.where(f != k)[0], np.where(f == k)[0]
    mdl, kap = fit_head(tr), fit_kappa(tr)
    for t in te:
        base = mdl.predict_proba(C_pr[[t]])[0].argmax()
        flips = []
        for j in range(len(FEATS)):
            X = np.repeat(C_pr[[t]], S, 0); X[:, j] += kap * dev_s[t, :, j]
            flips.append(np.mean(mdl.predict(X) != base))
        flips = np.array(flips); top = np.argsort(-flips)[:2]
        expl.append(dict(patient=order[t], label=classes[y[t]], pred=classes[base],
                         p_flip_any=np.mean(P['pp_cal'][t].argmax() != base),
                         doubt1=FEATS[top[0]], doubt1_flip=flips[top[0]], doubt2=FEATS[top[1]], doubt2_flip=flips[top[1]]))
ex = pd.DataFrame(expl)
w = ex[ex.label != ex.pred]
print(f'\nMisdiagnosed ({len(w)}) with their main source of doubt:')
print(w[['patient', 'label', 'pred', 'doubt1', 'doubt1_flip', 'doubt2', 'doubt2_flip']].round(2).to_string(index=False))
print('\nPatients with a measurement that flips the decision in >= 20% of its samples:',
      f'wrong {np.mean(w.doubt1_flip >= 0.2) * 100:.0f}% vs correct {np.mean(ex[ex.label == ex.pred].doubt1_flip >= 0.2) * 100:.0f}%')

res.to_csv(f'{OUT}/step8_results.csv', index=False)
ex.to_csv(f'{OUT}/doubt_per_patient.csv', index=False)
for m in MODES:
    np.save(f'{OUT}/oof_probs_{m}.npy', P[m]); np.save(f'{OUT}/sets_LAC_{m}.npy', SET[m])
print('\nSaved to', OUT)


### Paired bootstrap (CPU)
Confidence intervals for Section 4.2.

In [ ]:
import os, numpy as np, pandas as pd
from scipy.stats import fisher_exact, beta
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
S8 = f'{DRIVE}/step8_patient_uncertainty'
SEED, B, FLAG = 42, 2000, 0.20

folds = pd.read_csv(f'{DRIVE}/folds.csv')
classes = np.array(sorted(folds.label.unique()))
y = np.searchsorted(classes, folds.label.values)
n, K = len(y), len(classes)
P = {m: np.load(f'{S8}/oof_probs_{m}.npy') for m in ('plain', 'pp_raw', 'pp_cal')}
ex = pd.read_csv(f'{S8}/doubt_per_patient.csv').set_index('patient').loc[folds.patient]

def ece(Pm, yy, bins=15):
    conf, acc = Pm.max(1), (Pm.argmax(1) == yy).astype(float)
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

NAMES = ['bal_acc', 'AUC', 'NLL', 'Brier', 'ECE', 'err_detect_AUROC']
def metrics(Pm, i):
    yy, Q = y[i], Pm[i]
    wrong = Q.argmax(1) != yy
    return np.array([balanced_accuracy_score(yy, Q.argmax(1)),
                     roc_auc_score(yy, Q, multi_class='ovr', labels=range(K)),
                     -np.mean(np.log(np.clip(Q[np.arange(len(yy)), yy], 1e-12, 1))),
                     np.mean(((Q - np.eye(K)[yy]) ** 2).sum(1)),
                     ece(Q, yy),
                     roc_auc_score(wrong, 1 - Q.max(1)) if 0 < wrong.sum() < len(yy) else np.nan])

rng = np.random.default_rng(SEED)
boots = [i for i in (rng.integers(0, n, n) for _ in range(B)) if len(np.unique(y[i])) == K]
pt = {m: metrics(P[m], np.arange(n)) for m in P}
bs = {m: np.array([metrics(P[m], i) for i in boots]) for m in P}

pd.set_option('display.width', 200)
print('Point estimates [95% CI]:')
rows = []
for m in P:
    lo, hi = np.nanpercentile(bs[m], [2.5, 97.5], axis=0)
    rows.append({'model': m, **{k: f'{pt[m][j]:.3f} [{lo[j]:.3f}, {hi[j]:.3f}]' for j, k in enumerate(NAMES)}})
print(pd.DataFrame(rows).to_string(index=False))

print('\nPaired differences (model minus plain), 95% CI; * = CI excludes 0')
better = {'bal_acc': +1, 'AUC': +1, 'NLL': -1, 'Brier': -1, 'ECE': -1, 'err_detect_AUROC': +1}
drows = []
for m in ('pp_raw', 'pp_cal'):
    d = bs[m] - bs['plain']
    for j, k in enumerate(NAMES):
        lo, hi = np.nanpercentile(d[:, j], [2.5, 97.5])
        sig = '*' if lo > 0 or hi < 0 else ''
        direction = 'better' if np.sign(pt[m][j] - pt['plain'][j]) == better[k] else 'worse'
        drows.append(dict(model=m, metric=k, diff=pt[m][j] - pt['plain'][j], lo=lo, hi=hi, sig=sig, direction=direction,
                          p_better=np.nanmean(np.sign(d[:, j]) == better[k])))
D = pd.DataFrame(drows)
print(D.round(3).to_string(index=False))

wrong = (ex.label != ex.pred).values
flag = (ex.doubt1_flip >= FLAG).values
a, b_ = (flag & wrong).sum(), (~flag & wrong).sum()
c, d_ = (flag & ~wrong).sum(), (~flag & ~wrong).sum()
def cp(k, m):
    return (beta.ppf(0.025, k, m - k + 1) if k > 0 else 0.0, beta.ppf(0.975, k + 1, m - k) if k < m else 1.0)
odds, pv = fisher_exact([[a, b_], [c, d_]], alternative='greater')
print(f'\nFlip flag (>= {FLAG:.0%} of samples flip the decision):')
print(f'  sensitivity (wrong flagged)  = {a}/{a + b_} = {a / (a + b_):.2f}  95% CI {np.round(cp(a, a + b_), 2)}')
print(f'  false-flag rate (correct)    = {c}/{c + d_} = {c / (c + d_):.2f}  95% CI {np.round(cp(c, c + d_), 2)}')
print(f'  precision (flagged are wrong) = {a}/{a + c} = {a / max(a + c, 1):.2f}')
print(f'  Fisher exact (one-sided) p = {pv:.2g}, odds ratio = {odds:.1f}')
print(f'  AUROC of max flip fraction for detecting errors = {roc_auc_score(wrong, ex.doubt1_flip.values):.3f}')

D.to_csv(f'{S8}/step8b_paired_differences.csv', index=False)
print('\nSaved to', S8)


## Part 5 — External test on M&Ms-1 (Section 3.8, results in 4.3–4.4)
*Why does the rule fail at new hospitals?* The whole pipeline is frozen on ACDC and run on 271 DCM/HCM/NOR patients from four vendors. Each error is attributed to segmentation or to the rule by re-running the rule on reference measurements. The last cells explain the rule-driven errors: DCM phenotypes, centre 3, and the patients whose labels have no ACDC counterpart.

### Locked external test (GPU, ~30–40 min, resumable)
Segmentation, measurements and samples for every external patient.

In [ ]:
import os, glob, zipfile, io, time, re
import numpy as np, pandas as pd, nibabel as nib, torch, torch.nn as nn, torch.nn.functional as F
from scipy.ndimage import distance_transform_edt
from scipy.stats import spearmanr, fisher_exact, beta
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
STEP1 = f'{DRIVE}/step1_unet'
S8 = f'{DRIVE}/step8_patient_uncertainty'
OUT = f'{DRIVE}/step9_mnms_external'
LOCAL = '/content/mnms'
os.makedirs(OUT, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
SEED, SIZE, WIDTH, S, ALPHA, FLAG, B = 42, 224, 32, 20, 0.10, 0.20, 2000
PRIMARY = ['DCM', 'HCM', 'NOR']
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED); rng = np.random.default_rng(SEED)
print('device:', dev)

folds = pd.read_csv(f'{DRIVE}/folds.csv')
order = folds.patient.values
classes = np.array(sorted(folds.label.unique()))
y_acdc = np.searchsorted(classes, folds.label.values)
K = len(classes)
rf = pd.read_csv(f'{DRIVE}/step3_challenge_baseline/rich_features.csv')
FEATS = [c for c in rf.columns if c not in ('patient', 'source')]
A_gt = rf[rf.source == 'gt'].set_index('patient').loc[order, FEATS].values
A_pr = rf[rf.source == 'pred'].set_index('patient').loc[order, FEATS].values
A_cs = np.load(f'{S8}/concept_samples.npz')['CS']
head = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced')),
                    {'logisticregression__C': [0.01, 0.1, 1, 10, 100]},
                    cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy').fit(A_gt, y_acdc).best_estimator_
KAPPA = float(np.median(np.abs(A_pr - A_gt) / (A_cs.std(1) + 1e-9)) / 0.6745)
P_oof = np.load(f'{S8}/oof_probs_plain.npy')
cal = 1 - P_oof[np.arange(len(y_acdc)), y_acdc]
QHAT = np.quantile(cal, min(1, np.ceil((len(cal) + 1) * (1 - ALPHA)) / len(cal)), method='higher')
pidx = np.searchsorted(classes, PRIMARY)
print(f'locked: kappa = {KAPPA:.2f}, conformal qhat = {QHAT:.3f}, primary classes {PRIMARY} -> idx {pidx}')

ZIP = (glob.glob('/content/drive/MyDrive/**/M&M_dataset.zip', recursive=True) +
       glob.glob('/content/drive/MyDrive/**/M&M*.zip', recursive=True))[0]
z = zipfile.ZipFile(ZIP)
names = z.namelist()
meta_name = [n for n in names if 'MnMs_Dataset_information' in n and n.lower().endswith('.csv')][0]
meta = pd.read_csv(io.BytesIO(z.read(meta_name)))
col = lambda *keys: next(c for c in meta.columns if any(k.lower() == c.lower().strip() for k in keys))
C_CODE, C_PATH, C_ED, C_ES = col('External code'), col('Pathology'), col('ED'), col('ES')
C_VEND = next((c for c in meta.columns if c.lower().strip() in ('vendorname', 'vendor name')), col('Vendor'))
meta[C_CODE] = meta[C_CODE].astype(str).str.strip()
img_of = {re.sub(r'_sa\.nii(\.gz)?$', '', os.path.basename(n)): n for n in names if re.search(r'_sa\.nii(\.gz)?$', n)}
gt_of = {re.sub(r'_sa_gt\.nii(\.gz)?$', '', os.path.basename(n)): n for n in names if re.search(r'_sa_gt\.nii(\.gz)?$', n)}
M = meta[meta[C_CODE].isin(gt_of) & meta[C_CODE].isin(img_of) & meta[C_PATH].isin(PRIMARY)].reset_index(drop=True)
print(f'M&Ms primary population with ground truth: {len(M)}')
print(pd.crosstab(M[C_VEND], M[C_PATH]).to_string())

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(s, c=32, n=4, p=0.1):
        super().__init__()
        s.e1, s.e2, s.e3, s.e4 = block(1, c), block(c, 2 * c), block(2 * c, 4 * c), block(4 * c, 8 * c)
        s.b = block(8 * c, 16 * c)
        s.drop = nn.Dropout2d(p)
        s.u4, s.d4 = nn.ConvTranspose2d(16 * c, 8 * c, 2, 2), block(16 * c, 8 * c)
        s.u3, s.d3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, 2), block(8 * c, 4 * c)
        s.u2, s.d2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, 2), block(4 * c, 2 * c)
        s.u1, s.d1 = nn.ConvTranspose2d(2 * c, c, 2, 2), block(2 * c, c)
        s.out = nn.Conv2d(c, n, 1)

    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(F.max_pool2d(e1, 2)); e3 = s.e3(F.max_pool2d(e2, 2)); e4 = s.e4(F.max_pool2d(e3, 2))
        b = s.drop(s.b(F.max_pool2d(e4, 2)))
        d4 = s.d4(torch.cat([s.u4(b), e4], 1)); d3 = s.d3(torch.cat([s.u3(d4), e3], 1))
        d2 = s.d2(torch.cat([s.u2(d3), e2], 1)); d1 = s.d1(torch.cat([s.u1(d2), e1], 1))
        return s.out(d1)

def norm(v):
    lo, hi = np.percentile(v, [1, 99])
    v = np.clip(v, lo, hi)
    return ((v - v.mean()) / (v.std() + 1e-8)).astype(np.float32)

@torch.no_grad()
def seg(net, v, stochastic):
    H, W, Z = v.shape
    x = torch.from_numpy(np.ascontiguousarray(v.transpose(2, 0, 1)))[:, None].to(dev)
    x = F.interpolate(x, size=(SIZE, SIZE), mode='bilinear', align_corners=False)
    fl = []
    if stochastic:
        g = float(np.exp(rng.normal(0, 0.1)))
        x = torch.sign(x) * x.abs() ** g
        fl = [d for d in (2, 3) if rng.random() < 0.5]
        if fl: x = torch.flip(x, fl)
    p = net(x).float().softmax(1)
    if fl: p = torch.flip(p, fl)
    p = F.interpolate(p, size=(H, W), mode='bilinear', align_corners=False)
    return p.argmax(1).permute(1, 2, 0).cpu().numpy().astype(np.uint8)

def thickness_per_slice(myo, zooms):
    out = []
    for k in range(myo.shape[2]):
        m = myo[:, :, k]
        if m.sum() >= 10:
            out.append(2 * distance_transform_edt(m, sampling=zooms[:2]).max() - np.mean(zooms[:2]))
    return np.array(out) if out else np.array([0.0])

def rich_features(lab_ed, lab_es, z_ed, z_es):
    ved, ves = np.prod(z_ed) / 1000, np.prod(z_es) / 1000
    v = {f'{s}_{ph}': (lab == c).sum() * vv for ph, lab, vv in (('ED', lab_ed, ved), ('ES', lab_es, ves))
         for s, c in (('RV', 1), ('MYO', 2), ('LV', 3))}
    t_ed, t_es = thickness_per_slice(lab_ed == 2, z_ed), thickness_per_slice(lab_es == 2, z_es)
    eps = 1e-6
    return dict(**v,
                LV_EF=100 * (v['LV_ED'] - v['LV_ES']) / max(v['LV_ED'], eps),
                RV_EF=100 * (v['RV_ED'] - v['RV_ES']) / max(v['RV_ED'], eps),
                LV_mass=1.05 * v['MYO_ED'],
                RV_LV_ED=v['RV_ED'] / max(v['LV_ED'], eps), RV_LV_ES=v['RV_ES'] / max(v['LV_ES'], eps),
                MYO_LV_ED=v['MYO_ED'] / max(v['LV_ED'], eps), MYO_LV_ES=v['MYO_ES'] / max(v['LV_ES'], eps),
                WT_ED_max=t_ed.max(), WT_ED_mean=t_ed.mean(), WT_ED_std=t_ed.std(), WT_ED_min=t_ed.min(),
                WT_ES_max=t_es.max(), WT_ES_mean=t_es.mean(), WT_ES_std=t_es.std(), WT_ES_min=t_es.min())

REMAP = np.array([0, 3, 2, 1], np.uint8)
def dice(a, b, c):
    a, b = a == c, b == c
    return 2 * (a & b).sum() / max(a.sum() + b.sum(), 1)

CACHE = f'{OUT}/mnms_concepts.npz'
done = dict(np.load(CACHE, allow_pickle=True)) if os.path.exists(CACHE) else {}
res = done.get('res', np.array([], dtype=object)).tolist() if done else []
seen = {r['code'] for r in res}
nets = []
for k in range(5):
    net = UNet(c=WIDTH).to(dev)
    net.load_state_dict(torch.load(f'{STEP1}/unet_fold{k}.pt', map_location=dev))
    net.eval(); net.drop.train()
    nets.append(net)
t0 = time.time()
for i, r in M.iterrows():
    code = r[C_CODE]
    if code in seen: continue
    fi, fg = (z.extract(img_of[code], LOCAL), z.extract(gt_of[code], LOCAL))
    im, gt = nib.load(fi), nib.load(fg)
    zooms = tuple(float(v) for v in im.header.get_zooms()[:3])
    I, G = im.get_fdata(), np.rint(gt.get_fdata()).astype(np.uint8)
    ed, es = int(r[C_ED]), int(r[C_ES])
    if G[..., ed].max() == 0 or G[..., es].max() == 0:
        lab_t = [t for t in range(G.shape[3]) if G[..., t].max() > 0]
        ed, es = lab_t[0], lab_t[-1]
    vols = [norm(I[..., ed]), norm(I[..., es])]
    gts = [REMAP[G[..., ed]], REMAP[G[..., es]]]
    net = nets[i % 5]
    net.drop.eval(); det = [seg(net, v, False) for v in vols]; net.drop.train()
    samp = []
    for s in range(S):
        ft = rich_features(seg(net, vols[0], True), seg(net, vols[1], True), zooms, zooms)
        samp.append([ft[c] for c in FEATS])
    fg_ = rich_features(gts[0], gts[1], zooms, zooms); fp_ = rich_features(det[0], det[1], zooms, zooms)
    res.append(dict(code=code, vendor=r[C_VEND], label=r[C_PATH],
                    gt=np.array([fg_[c] for c in FEATS]), pred=np.array([fp_[c] for c in FEATS]), samples=np.array(samp),
                    dice={f'{nm}_{ph}': dice(det[j], gts[j], c) for j, ph in enumerate(('ED', 'ES'))
                          for nm, c in (('RV', 1), ('MYO', 2), ('LV', 3))}))
    os.remove(fi); os.remove(fg)
    if len(res) % 20 == 0 or i == len(M) - 1:
        np.savez(CACHE, res=np.array(res, dtype=object))
        print(f'{len(res)}/{len(M)} patients ({time.time() - t0:.0f}s)')
np.savez(CACHE, res=np.array(res, dtype=object))

R = pd.DataFrame(res)
n = len(R)
Cg, Cp, CS = np.stack(list(R['gt'])), np.stack(list(R['pred'])), np.stack(list(R['samples']))
yv = np.searchsorted(PRIMARY, R.label.values)
vend = R.vendor.values
dv = CS - CS.mean(1, keepdims=True)

def restrict(P5):
    Q = P5[:, pidx]; return Q / Q.sum(1, keepdims=True)

P5 = head.predict_proba(Cp)
P3 = restrict(P5)
P3_gt = restrict(head.predict_proba(Cg))
pred3 = P3.argmax(1)
wrong = pred3 != yv

flips = np.zeros((n, len(FEATS)))
for t in range(n):
    for j in range(len(FEATS)):
        X = np.repeat(Cp[[t]], S, 0); X[:, j] += KAPPA * dv[t, :, j]
        flips[t, j] = np.mean(restrict(head.predict_proba(X)).argmax(1) != pred3[t])
fmax = flips.max(1); doubt = np.array(FEATS)[flips.argmax(1)]
flag = fmax >= FLAG
SET = (1 - P5) <= QHAT
cover = SET[np.arange(n), pidx[yv]]

D = pd.DataFrame(list(R['dice']))
print('\nSegmentation Dice (deterministic) per vendor:')
print(D.assign(vendor=vend).groupby('vendor').mean().round(3).to_string())

def summary(m):
    yy, P, w, fl = yv[m], P3[m], wrong[m], flag[m]
    out = dict(n=m.sum(), bal_acc=balanced_accuracy_score(yy, P.argmax(1)),
               bal_acc_refmeas=balanced_accuracy_score(yy, P3_gt[m].argmax(1)),
               AUC=roc_auc_score(yy, P, multi_class='ovr', labels=range(3)) if len(np.unique(yy)) == 3 else np.nan,
               errors=w.sum(), flag_sens=fl[w].mean() if w.any() else np.nan, false_flag=fl[~w].mean(),
               AUROC_flip=roc_auc_score(w, fmax[m]) if 0 < w.sum() < m.sum() else np.nan,
               AUROC_conf=roc_auc_score(w, 1 - P.max(1)) if 0 < w.sum() < m.sum() else np.nan,
               conformal_cov=cover[m].mean(), set_size=SET[m].sum(1).mean())
    return out
rows = [dict(vendor='ALL', **summary(np.ones(n, bool)))] + [dict(vendor=v, **summary(vend == v)) for v in sorted(set(vend))]
T = pd.DataFrame(rows)
pd.set_option('display.width', 220)
print('\nExternal results (primary: DCM / HCM / NOR):')
print(T.round(3).to_string(index=False))
print('\nConfusion (rows = truth):', PRIMARY); print(confusion_matrix(yv, pred3, labels=range(3)))

a, b_, c, d_ = (flag & wrong).sum(), (~flag & wrong).sum(), (flag & ~wrong).sum(), (~flag & ~wrong).sum()
odds, pv = fisher_exact([[a, b_], [c, d_]], alternative='greater')
def cp(k, m):
    return (beta.ppf(0.025, k, m - k + 1) if k > 0 else 0.0, beta.ppf(0.975, k + 1, m - k) if k < m else 1.0)
print(f'\nFlip flag: sensitivity {a}/{a + b_} CI {np.round(cp(a, a + b_), 2)} | false-flag {c}/{c + d_} CI {np.round(cp(c, c + d_), 2)}'
      f' | Fisher p = {pv:.2g}, OR = {odds:.1f}')

bs = []
for _ in range(B):
    i = rng.integers(0, n, n)
    if 0 < wrong[i].sum() < n:
        bs.append(roc_auc_score(wrong[i], fmax[i]) - roc_auc_score(wrong[i], 1 - P3[i].max(1)))
lo, hi = np.percentile(bs, [2.5, 97.5])
print(f'Paired AUROC difference (flip flag minus confidence): {T.AUROC_flip[0] - T.AUROC_conf[0]:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]')

print('\nSpearman: per-patient sample SD vs real |pred - gt| (does uncertainty still track error?)')
sd = CS.std(1)
for c_ in ('LV_EF', 'RV_EF', 'LV_ED', 'RV_ED', 'MYO_ED', 'RV_LV_ED', 'MYO_LV_ED'):
    j = FEATS.index(c_); r_, p_ = spearmanr(sd[:, j], np.abs(Cp[:, j] - Cg[:, j]))
    print(f'  {c_:10s}: rho = {r_:+.2f}  p = {p_:.2g}')

print('\nMain source of doubt among errors:'); print(pd.Series(doubt[wrong]).value_counts().head(6).to_string())

T.to_csv(f'{OUT}/step9_results.csv', index=False)
pd.DataFrame(dict(code=R.code, vendor=vend, label=R.label, pred=np.array(PRIMARY)[pred3], flip_max=fmax, doubt=doubt,
                  flag=flag, in_set=cover, **{f'p_{c_}': P3[:, j] for j, c_ in enumerate(PRIMARY)})).to_csv(f'{OUT}/per_patient.csv', index=False)
D.assign(code=R.code, vendor=vend).to_csv(f'{OUT}/dice.csv', index=False)
print('\nSaved to', OUT)


### Error attribution (CPU)
Segmentation-driven vs rule-driven errors.

In [ ]:
import os, numpy as np, pandas as pd
from scipy.special import logsumexp
from scipy.stats import fisher_exact, beta
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
S8, OUT = f'{DRIVE}/step8_patient_uncertainty', f'{DRIVE}/step9_mnms_external'
SEED, S, FLAG, B = 42, 20, 0.20, 2000
PRIMARY = ['DCM', 'HCM', 'NOR']
rng = np.random.default_rng(SEED)

folds = pd.read_csv(f'{DRIVE}/folds.csv')
order = folds.patient.values
classes = np.array(sorted(folds.label.unique()))
y_acdc = np.searchsorted(classes, folds.label.values)
rf = pd.read_csv(f'{DRIVE}/step3_challenge_baseline/rich_features.csv')
FEATS = [c for c in rf.columns if c not in ('patient', 'source')]
A_gt = rf[rf.source == 'gt'].set_index('patient').loc[order, FEATS].values
A_pr = rf[rf.source == 'pred'].set_index('patient').loc[order, FEATS].values
A_cs = np.load(f'{S8}/concept_samples.npz')['CS']
head = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced')),
                    {'logisticregression__C': [0.01, 0.1, 1, 10, 100]},
                    cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy').fit(A_gt, y_acdc).best_estimator_
KAPPA = float(np.median(np.abs(A_pr - A_gt) / (A_cs.std(1) + 1e-9)) / 0.6745)
pidx = np.searchsorted(classes, PRIMARY)

def p3(X):
    L = head.decision_function(X)[:, pidx]
    return np.exp(L - logsumexp(L, 1, keepdims=True))

R = pd.DataFrame(list(np.load(f'{OUT}/mnms_concepts.npz', allow_pickle=True)['res']))
n = len(R)
Cg, Cp, CS = np.stack(list(R['gt'])), np.stack(list(R['pred'])), np.stack(list(R['samples']))
yv = np.searchsorted(PRIMARY, R.label.values); vend = R.vendor.values
dv = CS - CS.mean(1, keepdims=True)
Pp, Pg = p3(Cp), p3(Cg)
pred, ref = Pp.argmax(1), Pg.argmax(1)
wrong, ref_ok = pred != yv, ref == yv
print(f'n = {n} | kappa = {KAPPA:.2f}')
print(f'bal acc (predicted measurements) = {balanced_accuracy_score(yv, pred):.3f} | '
      f'bal acc (reference measurements) = {balanced_accuracy_score(yv, ref):.3f}')

flips = np.zeros((n, len(FEATS)))
for t in range(n):
    for j in range(len(FEATS)):
        X = np.repeat(Cp[[t]], S, 0); X[:, j] += KAPPA * dv[t, :, j]
        flips[t, j] = np.mean(p3(X).argmax(1) != pred[t])
fmax = flips.max(1); flag = fmax >= FLAG; conf = 1 - Pp.max(1)
doubt = np.array(FEATS)[flips.argmax(1)]

typ = np.where(~wrong, 'CORRECT', np.where(ref_ok, 'SEG', 'RULE'))
rescued = (~ref_ok) & (~wrong)
print('\nError types (pred wrong):')
print(pd.crosstab(pd.Series(vend, name='vendor'), pd.Series(typ, name='type'), margins=True).to_string())
print(f'(also: {rescued.sum()} patients wrong with reference measurements but right with predicted ones)')

def cp(k, m):
    return (beta.ppf(0.025, k, m - k + 1) if k > 0 else 0.0, beta.ppf(0.975, k + 1, m - k) if k < m else 1.0)
ok = typ == 'CORRECT'
print(f'\nFalse-flag rate on correct patients: {flag[ok].sum()}/{ok.sum()} = {flag[ok].mean():.2f} CI {np.round(cp(flag[ok].sum(), ok.sum()), 2)}')
rows = []
for tname in ('SEG', 'RULE'):
    m = typ == tname
    if m.sum() == 0: continue
    sel = m | ok; w = m[sel]
    odds, pv = fisher_exact([[flag[m].sum(), (~flag[m]).sum()], [flag[ok].sum(), (~flag[ok]).sum()]], alternative='greater')
    au_f, au_c = roc_auc_score(w, fmax[sel]), roc_auc_score(w, conf[sel])
    bs = []
    for _ in range(B):
        i = rng.choice(np.where(sel)[0], sel.sum())
        ww = m[i]
        if 0 < ww.sum() < len(i):
            bs.append(roc_auc_score(ww, fmax[i]) - roc_auc_score(ww, conf[i]))
    lo, hi = np.percentile(bs, [2.5, 97.5])
    rows.append(dict(type=tname, n=m.sum(), flag_sens=flag[m].mean(), sens_CI=np.round(cp(flag[m].sum(), m.sum()), 2),
                     Fisher_p=pv, OR=odds, AUROC_flip=au_f, AUROC_conf=au_c, diff=au_f - au_c, diff_lo=lo, diff_hi=hi))
T = pd.DataFrame(rows)
pd.set_option('display.width', 220)
print('\nDoes the flag catch each error type? (vs correct patients)')
print(T.round(3).to_string(index=False))

print('\nWhere do RULE errors go? (truth -> decision even with reference measurements)')
mr = typ == 'RULE'
print(pd.crosstab(pd.Series(np.array(PRIMARY)[yv[mr]], name='truth'), pd.Series(np.array(PRIMARY)[ref[mr]], name='ref decision')).to_string())
print('\nMedian reference LV_EF / LV_ED by class: ACDC vs M&Ms (is the population different?)')
iE, iV = FEATS.index('LV_EF'), FEATS.index('LV_ED')
for c in PRIMARY:
    a = classes[y_acdc] == c; b = R.label.values == c
    print(f'  {c}: LV_EF {np.median(A_gt[a, iE]):5.1f} vs {np.median(Cg[b, iE]):5.1f} | '
          f'LV_ED {np.median(A_gt[a, iV]):6.1f} vs {np.median(Cg[b, iV]):6.1f} mL')
print('\nMain source of doubt among SEG errors:'); print(pd.Series(doubt[typ == 'SEG']).value_counts().head(5).to_string())

pd.DataFrame(dict(code=R.code, vendor=vend, label=R.label, pred=np.array(PRIMARY)[pred], ref_decision=np.array(PRIMARY)[ref],
                  type=typ, flip_max=fmax, flag=flag, doubt=doubt)).to_csv(f'{OUT}/error_types.csv', index=False)
T.to_csv(f'{OUT}/flag_by_error_type.csv', index=False)
print('\nSaved to', OUT)


### ED/ES rule and final external results (CPU)
Table 3; both versions are reported.

In [ ]:
import os, io, glob, zipfile, numpy as np, pandas as pd
from scipy.special import logsumexp
from scipy.stats import fisher_exact, beta, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
S8, OUT = f'{DRIVE}/step8_patient_uncertainty', f'{DRIVE}/step9_mnms_external'
SEED, S, FLAG, ALPHA, B = 42, 20, 0.20, 0.10, 2000
PRIMARY = ['DCM', 'HCM', 'NOR']
rng = np.random.default_rng(SEED)

folds = pd.read_csv(f'{DRIVE}/folds.csv')
order = folds.patient.values
classes = np.array(sorted(folds.label.unique()))
y_acdc = np.searchsorted(classes, folds.label.values)
rf = pd.read_csv(f'{DRIVE}/step3_challenge_baseline/rich_features.csv')
FEATS = [c for c in rf.columns if c not in ('patient', 'source')]
A_gt = rf[rf.source == 'gt'].set_index('patient').loc[order, FEATS].values
A_pr = rf[rf.source == 'pred'].set_index('patient').loc[order, FEATS].values
A_cs = np.load(f'{S8}/concept_samples.npz')['CS']
head = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced')),
                    {'logisticregression__C': [0.01, 0.1, 1, 10, 100]},
                    cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy').fit(A_gt, y_acdc).best_estimator_
KAPPA = float(np.median(np.abs(A_pr - A_gt) / (A_cs.std(1) + 1e-9)) / 0.6745)
P_oof = np.load(f'{S8}/oof_probs_plain.npy')
cal = 1 - P_oof[np.arange(len(y_acdc)), y_acdc]
QHAT = np.quantile(cal, min(1, np.ceil((len(cal) + 1) * (1 - ALPHA)) / len(cal)), method='higher')
pidx = np.searchsorted(classes, PRIMARY)

R = pd.DataFrame(list(np.load(f'{OUT}/mnms_concepts.npz', allow_pickle=True)['res']))
z = zipfile.ZipFile(glob.glob('/content/drive/MyDrive/**/M&M*.zip', recursive=True)[0])
meta = pd.read_csv(io.BytesIO(z.read([n for n in z.namelist() if 'MnMs_Dataset_information' in n][0])))
meta['External code'] = meta['External code'].astype(str).str.strip()
R['centre'] = R.code.map(meta.set_index('External code')['Centre'])
n = len(R)
Cg, Cp, CS = np.stack(list(R['gt'])), np.stack(list(R['pred'])), np.stack(list(R['samples']))
F = {c: j for j, c in enumerate(FEATS)}

def swap_phases(X):
    """swap ED <-> ES in a (..., 21) feature array and recompute the derived features."""
    X = X.copy()
    for a in ('RV', 'MYO', 'LV'):
        X[..., [F[f'{a}_ED'], F[f'{a}_ES']]] = X[..., [F[f'{a}_ES'], F[f'{a}_ED']]]
    for s in ('max', 'mean', 'std', 'min'):
        X[..., [F[f'WT_ED_{s}'], F[f'WT_ES_{s}']]] = X[..., [F[f'WT_ES_{s}'], F[f'WT_ED_{s}']]]
    e = 1e-6
    g = lambda c: X[..., F[c]]
    X[..., F['LV_EF']] = 100 * (g('LV_ED') - g('LV_ES')) / np.maximum(g('LV_ED'), e)
    X[..., F['RV_EF']] = 100 * (g('RV_ED') - g('RV_ES')) / np.maximum(g('RV_ED'), e)
    X[..., F['LV_mass']] = 1.05 * g('MYO_ED')
    X[..., F['RV_LV_ED']] = g('RV_ED') / np.maximum(g('LV_ED'), e); X[..., F['RV_LV_ES']] = g('RV_ES') / np.maximum(g('LV_ES'), e)
    X[..., F['MYO_LV_ED']] = g('MYO_ED') / np.maximum(g('LV_ED'), e); X[..., F['MYO_LV_ES']] = g('MYO_ES') / np.maximum(g('LV_ES'), e)
    return X

sw = Cp[:, F['LV_ED']] < Cp[:, F['LV_ES']]
Cg[sw], Cp[sw], CS[sw] = swap_phases(Cg[sw]), swap_phases(Cp[sw]), swap_phases(CS[sw])
print(f'ED/ES swapped by the rule: {sw.sum()}/{n} patients')
print(pd.crosstab(R.centre, sw, rownames=['centre'], colnames=['swapped']).to_string())
print(f'negative reference LV_EF after the rule: {(Cg[:, F["LV_EF"]] < 0).sum()}')
print('\nMedian reference LV_EF by class and centre (sanity check):')
print(pd.DataFrame(dict(label=R.label, centre=R.centre, LV_EF=Cg[:, F['LV_EF']])).pivot_table(index='label', columns='centre', values='LV_EF', aggfunc='median').round(1).to_string())

def p3(X):
    L = head.decision_function(X)[:, pidx]
    return np.exp(L - logsumexp(L, 1, keepdims=True))
yv = np.searchsorted(PRIMARY, R.label.values); vend = R.vendor.values; cen = R.centre.values
dv = CS - CS.mean(1, keepdims=True)
Pp, Pg = p3(Cp), p3(Cg)
P5 = head.predict_proba(Cp)
pred, ref = Pp.argmax(1), Pg.argmax(1)
wrong, ref_ok = pred != yv, ref == yv
flips = np.zeros((n, len(FEATS)))
for t in range(n):
    for j in range(len(FEATS)):
        X = np.repeat(Cp[[t]], S, 0); X[:, j] += KAPPA * dv[t, :, j]
        flips[t, j] = np.mean(p3(X).argmax(1) != pred[t])
fmax = flips.max(1); flag = fmax >= FLAG; conf = 1 - Pp.max(1)
doubt = np.array(FEATS)[flips.argmax(1)]
cover = ((1 - P5) <= QHAT)[np.arange(n), pidx[yv]]
setsz = ((1 - P5) <= QHAT).sum(1)
typ = np.where(~wrong, 'CORRECT', np.where(ref_ok, 'SEG', 'RULE'))

def summary(m):
    yy, w = yv[m], wrong[m]
    return dict(n=m.sum(), bal_acc=balanced_accuracy_score(yy, pred[m]), bal_acc_refmeas=balanced_accuracy_score(yy, ref[m]),
                AUC=roc_auc_score(yy, Pp[m], multi_class='ovr', labels=range(3)) if len(np.unique(yy)) == 3 else np.nan,
                errors=w.sum(), SEG=(typ[m] == 'SEG').sum(), RULE=(typ[m] == 'RULE').sum(),
                flag_sens=flag[m][w].mean() if w.any() else np.nan, false_flag=flag[m][~w].mean(),
                AUROC_flip=roc_auc_score(w, fmax[m]) if 0 < w.sum() < m.sum() else np.nan,
                AUROC_conf=roc_auc_score(w, conf[m]) if 0 < w.sum() < m.sum() else np.nan,
                conformal_cov=cover[m].mean(), set_size=setsz[m].mean())
rows = [dict(subset='ALL', **summary(np.ones(n, bool)))]
rows += [dict(subset=v, **summary(vend == v)) for v in sorted(set(vend))]
rows += [dict(subset='ALL without centre 3 (sensitivity)', **summary(cen != 3))]
T = pd.DataFrame(rows)
pd.set_option('display.width', 240)
print('\nExternal results after the ED/ES rule (primary: DCM / HCM / NOR):')
print(T.round(3).to_string(index=False))
print('\nConfusion (rows = truth):', PRIMARY); print(confusion_matrix(yv, pred, labels=range(3)))

def cp(k, m):
    return (beta.ppf(0.025, k, m - k + 1) if k > 0 else 0.0, beta.ppf(0.975, k + 1, m - k) if k < m else 1.0)
ok = typ == 'CORRECT'
print(f'\nFalse-flag rate on correct patients: {flag[ok].sum()}/{ok.sum()} CI {np.round(cp(flag[ok].sum(), ok.sum()), 2)}')
for tname in ('SEG', 'RULE'):
    m = typ == tname
    if m.sum() == 0: continue
    sel = m | ok
    odds, pv = fisher_exact([[flag[m].sum(), (~flag[m]).sum()], [flag[ok].sum(), (~flag[ok]).sum()]], alternative='greater')
    bs = []
    for _ in range(B):
        i = rng.choice(np.where(sel)[0], sel.sum()); ww = m[i]
        if 0 < ww.sum() < len(i): bs.append(roc_auc_score(ww, fmax[i]) - roc_auc_score(ww, conf[i]))
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f'{tname:4s} errors n={m.sum():3d}: flag sens {flag[m].mean():.2f} CI {np.round(cp(flag[m].sum(), m.sum()), 2)} | '
          f'Fisher p = {pv:.2g}, OR = {odds:.1f} | AUROC flip {roc_auc_score(m[sel], fmax[sel]):.3f} vs conf '
          f'{roc_auc_score(m[sel], conf[sel]):.3f}, diff CI [{lo:+.3f}, {hi:+.3f}]')

print('\nSpearman: per-patient sample SD vs real |pred - gt|')
sd = CS.std(1)
for c in ('LV_EF', 'RV_EF', 'LV_ED', 'RV_ED', 'MYO_ED'):
    r_, p_ = spearmanr(sd[:, F[c]], np.abs(Cp[:, F[c]] - Cg[:, F[c]])); print(f'  {c:7s}: rho = {r_:+.2f}  p = {p_:.2g}')
print('\nMain source of doubt among SEG errors:'); print(pd.Series(doubt[typ == 'SEG']).value_counts().head(5).to_string())

T.to_csv(f'{OUT}/step9c_results.csv', index=False)
pd.DataFrame(dict(code=R.code, vendor=vend, centre=cen, label=R.label, swapped=sw, pred=np.array(PRIMARY)[pred],
                  ref_decision=np.array(PRIMARY)[ref], type=typ, flip_max=fmax, flag=flag, doubt=doubt,
                  in_set=cover)).to_csv(f'{OUT}/step9c_per_patient.csv', index=False)
print('\nSaved to', OUT)


### Shared objects for the analyses below (CPU)
Rebuilds once the locked objects of the previous step (rule, κ, external cohort after the ED/ES rule); reused in Parts 5 and 7.

In [ ]:
import os, io, re, glob, zipfile, numpy as np, pandas as pd, nibabel as nib
from scipy.ndimage import distance_transform_edt
from scipy.special import logsumexp
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
pd.set_option('display.width', 240)
PRIMARY = ['DCM', 'HCM', 'NOR']

def ext_fit(X, y, seed=42):
    """logistic rule, C chosen by inner 5-fold CV on balanced accuracy (same as every step)."""
    return GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced')),
                        {'logisticregression__C': [0.01, 0.1, 1, 10, 100]},
                        cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                        scoring='balanced_accuracy').fit(X, y).best_estimator_

def ext_swap(X, F):
    """swap ED <-> ES in a (..., 21) measurement array and recompute the derived measurements."""
    X = X.copy()
    for a in ('RV', 'MYO', 'LV'):
        X[..., [F[f'{a}_ED'], F[f'{a}_ES']]] = X[..., [F[f'{a}_ES'], F[f'{a}_ED']]]
    for s in ('max', 'mean', 'std', 'min'):
        X[..., [F[f'WT_ED_{s}'], F[f'WT_ES_{s}']]] = X[..., [F[f'WT_ES_{s}'], F[f'WT_ED_{s}']]]
    e, g = 1e-6, (lambda c: X[..., F[c]])
    X[..., F['LV_EF']] = 100 * (g('LV_ED') - g('LV_ES')) / np.maximum(g('LV_ED'), e)
    X[..., F['RV_EF']] = 100 * (g('RV_ED') - g('RV_ES')) / np.maximum(g('RV_ED'), e)
    X[..., F['LV_mass']] = 1.05 * g('MYO_ED')
    X[..., F['RV_LV_ED']] = g('RV_ED') / np.maximum(g('LV_ED'), e); X[..., F['RV_LV_ES']] = g('RV_ES') / np.maximum(g('LV_ES'), e)
    X[..., F['MYO_LV_ED']] = g('MYO_ED') / np.maximum(g('LV_ED'), e); X[..., F['MYO_LV_ES']] = g('MYO_ES') / np.maximum(g('LV_ES'), e)
    return X

def ext_rich_features(lab_ed, lab_es, z_ed, z_es):
    """the 21 measurements of Section 3.2 (identical to step 3). Labels: 1 RV, 2 MYO, 3 LV."""
    def thick(myo, zooms):
        out = [2 * distance_transform_edt(myo[:, :, k], sampling=zooms[:2]).max() - np.mean(zooms[:2])
               for k in range(myo.shape[2]) if myo[:, :, k].sum() >= 10]
        return np.array(out) if out else np.array([0.0])
    ved, ves = np.prod(z_ed) / 1000, np.prod(z_es) / 1000
    v = {f'{s}_{ph}': (lab == c).sum() * vv for ph, lab, vv in (('ED', lab_ed, ved), ('ES', lab_es, ves))
         for s, c in (('RV', 1), ('MYO', 2), ('LV', 3))}
    t_ed, t_es, eps = thick(lab_ed == 2, z_ed), thick(lab_es == 2, z_es), 1e-6
    return dict(**v, LV_EF=100 * (v['LV_ED'] - v['LV_ES']) / max(v['LV_ED'], eps),
                RV_EF=100 * (v['RV_ED'] - v['RV_ES']) / max(v['RV_ED'], eps), LV_mass=1.05 * v['MYO_ED'],
                RV_LV_ED=v['RV_ED'] / max(v['LV_ED'], eps), RV_LV_ES=v['RV_ES'] / max(v['LV_ES'], eps),
                MYO_LV_ED=v['MYO_ED'] / max(v['LV_ED'], eps), MYO_LV_ES=v['MYO_ES'] / max(v['LV_ES'], eps),
                WT_ED_max=t_ed.max(), WT_ED_mean=t_ed.mean(), WT_ED_std=t_ed.std(), WT_ED_min=t_ed.min(),
                WT_ES_max=t_es.max(), WT_ES_mean=t_es.mean(), WT_ES_std=t_es.std(), WT_ES_min=t_es.min())

_EXT = {}
def ext_load():
    if _EXT:
        return _EXT
    D = os.environ['ACDC_DRIVE']
    S8, S9 = f'{D}/step8_patient_uncertainty', f'{D}/step9_mnms_external'
    folds = pd.read_csv(f'{D}/folds.csv'); order = folds.patient.values
    classes = np.array(sorted(folds.label.unique())); y = np.searchsorted(classes, folds.label.values)
    rf = pd.read_csv(f'{D}/step3_challenge_baseline/rich_features.csv')
    FEATS = [c for c in rf.columns if c not in ('patient', 'source')]; F = {c: j for j, c in enumerate(FEATS)}
    A_gt = rf[rf.source == 'gt'].set_index('patient').loc[order, FEATS].values
    A_pr = rf[rf.source == 'pred'].set_index('patient').loc[order, FEATS].values
    A_cs = np.load(f'{S8}/concept_samples.npz')['CS']
    head = ext_fit(A_gt, y)
    KAPPA = float(np.median(np.abs(A_pr - A_gt) / (A_cs.std(1) + 1e-9)) / 0.6745)
    pidx = np.searchsorted(classes, PRIMARY)
    z = zipfile.ZipFile(glob.glob('/content/drive/MyDrive/**/M&M*.zip', recursive=True)[0])
    meta = pd.read_csv(io.BytesIO(z.read([n for n in z.namelist() if 'MnMs_Dataset_information' in n][0])))
    meta['External code'] = meta['External code'].astype(str).str.strip()
    R = pd.DataFrame(list(np.load(f'{S9}/mnms_concepts.npz', allow_pickle=True)['res']))
    R['centre'] = R.code.map(meta.set_index('External code')['Centre'])
    Cg, Cp, CS = np.stack(list(R['gt'])), np.stack(list(R['pred'])), np.stack(list(R['samples']))
    sw = Cp[:, F['LV_ED']] < Cp[:, F['LV_ES']]
    Cg[sw], Cp[sw], CS[sw] = ext_swap(Cg[sw], F), ext_swap(Cp[sw], F), ext_swap(CS[sw], F)
    def p3(X):
        L = head.decision_function(X)[:, pidx]
        return np.exp(L - logsumexp(L, 1, keepdims=True))
    yv = np.searchsorted(PRIMARY, R.label.values)
    pred, ref = p3(Cp).argmax(1), p3(Cg).argmax(1)
    wrong = pred != yv
    typ = np.where(~wrong, 'CORRECT', np.where(ref == yv, 'SEG', 'RULE'))
    _EXT.update({k: v for k, v in locals().items() if not k.startswith('_')})
    return _EXT

def ext_flip_max(E, S=20):
    """largest flip fraction per M&Ms-1 patient (Section 3.6), computed once."""
    if 'fmax' not in E:
        dv = E['CS'] - E['CS'].mean(1, keepdims=True)
        n, D = E['Cp'].shape
        fmax = np.zeros(n)
        for t in range(n):
            for j in range(D):
                X = np.repeat(E['Cp'][[t]], S, 0); X[:, j] += E['KAPPA'] * dv[t, :, j]
                fmax[t] = max(fmax[t], np.mean(E['p3'](X).argmax(1) != E['pred'][t]))
        E['fmax'] = fmax
    return E['fmax']

E = ext_load()
print(f"locked rule ready: kappa = {E['KAPPA']:.2f}; M&Ms-1 primary patients = {len(E['R'])}; "
      f"errors = {E['wrong'].sum()} (segmentation {np.sum(E['typ'] == 'SEG')}, rule {np.sum(E['typ'] == 'RULE')})")


### DCM phenotype strata and centre 3 (CPU)
Section 4.3 (new paragraph) and Section 4.4, item 2.

In [ ]:
E = ext_load(); FI, R, cen = E['F'], E['R'], E['R'].centre.values
EF, EDV = E['Cg'][:, FI['LV_EF']], E['Cg'][:, FI['LV_ED']]
nor, dcm = R.label.values == 'NOR', R.label.values == 'DCM'
EDV_ULN = np.percentile(EDV[nor], 95)
stratum = np.full(len(R), '', dtype=object)
stratum[dcm & (EF < 40) & (EDV > EDV_ULN)] = 'Classic DCM'
stratum[dcm & (EF >= 50)] = 'Preserved-function DCM'
stratum[dcm & (stratum == '')] = 'Intermediate DCM'
acdc_dcm = E['folds'].label.values == 'DCM'
print(f'LV EDV upper limit (95th pct of M&Ms-1 NOR): {EDV_ULN:.0f} mL')
print(f"ACDC DCM: median LV EF {np.median(E['A_gt'][acdc_dcm, FI['LV_EF']]):.1f} %, median LV EDV "
      f"{np.median(E['A_gt'][acdc_dcm, FI['LV_ED']]):.0f} mL")
rows = []
for s in ['Classic DCM', 'Intermediate DCM', 'Preserved-function DCM']:
    m = stratum == s
    rows.append(dict(stratum=s, n=m.sum(), median_EF=np.median(EF[m]), median_EDV=np.median(EDV[m]),
                     recall=np.mean(E['pred'][m] == 0), recall_refmeas=np.mean(E['ref'][m] == 0),
                     errors=E['wrong'][m].sum(), SEG=np.sum(E['typ'][m] == 'SEG'), RULE=np.sum(E['typ'][m] == 'RULE')))
print(pd.DataFrame(rows).round(2).to_string(index=False))
print(pd.crosstab(stratum[dcm], cen[dcm], rownames=['stratum'], colnames=['centre']).to_string())
c3 = dcm & (cen == 3)
print(f'Centre 3 DCM vs M&Ms-1 NOR: median LV EDV {np.median(EDV[c3]):.0f} vs {np.median(EDV[nor]):.0f} mL; '
      f'dilated: {np.mean(EDV[c3] > EDV_ULN) * 100:.0f} % of centre-3 DCM')
pd.DataFrame(rows).to_csv(f"{os.environ['ACDC_DRIVE']}/step9_mnms_external/step13_dcm_strata.csv", index=False)


### Patients without an ACDC counterpart class (CPU)
Section 4.3, secondary and descriptive.

In [ ]:
E = ext_load(); z, meta, FI, classes = E['z'], E['meta'], E['F'], E['classes']
REMAP = np.array([0, 3, 2, 1], np.uint8)
gt_of = {re.sub(r'_sa_gt\.nii(\.gz)?$', '', os.path.basename(n)): n for n in z.namelist() if re.search(r'_sa_gt\.nii(\.gz)?$', n)}
O = meta[meta['External code'].isin(gt_of) & ~meta['Pathology'].isin(PRIMARY)].reset_index(drop=True)
os.makedirs('/content/mnms_other', exist_ok=True)
X = []
for code in O['External code']:
    f = z.extract(gt_of[code], '/content/mnms_other')
    g = nib.load(f); zooms = tuple(float(v) for v in g.header.get_zooms()[:3])
    G = np.rint(g.get_fdata()).astype(np.uint8)
    lab_t = [t for t in range(G.shape[3]) if G[..., t].max() > 0]
    a, c = REMAP[G[..., lab_t[0]]], REMAP[G[..., lab_t[-1]]]
    if (a == 3).sum() < (c == 3).sum():
        a, c = c, a
    ft = ext_rich_features(a, c, zooms, zooms); X.append([ft[k] for k in E['FEATS']]); os.remove(f)
X = np.array(X); assigned = classes[E['head'].predict(X)]
print(f'{len(O)} patients outside DCM/HCM/NOR; rows = M&Ms-1 label, columns = ACDC class assigned')
print(pd.crosstab(O['Pathology'].values, assigned, margins=True).to_string())
print(pd.DataFrame(dict(label=O['Pathology'].values, LV_EF=X[:, FI['LV_EF']], RV_EF=X[:, FI['RV_EF']],
                        LV_EDV=X[:, FI['LV_ED']], RV_EDV=X[:, FI['RV_ED']], WT_ED_max=X[:, FI['WT_ED_max']]))
      .groupby('label').median().round(1).to_string())


## Part 6 — Few-shot recalibration (Section 3.9, results in 4.5)
*Can a few labelled local cases repair rule-driven errors?* Three class offsets are re-estimated from k labelled patients per class and compared with a full refit.

### Class offsets vs full refit (CPU)
Table 4.

In [ ]:
import os, numpy as np, pandas as pd
from scipy.special import logsumexp, log_softmax
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
OUT = f'{DRIVE}/step9_mnms_external'
SEED, KS, DRAWS, LAM, W_NEW = 42, (1, 3, 5, 10), 100, 1.0, 5.0
PRIMARY = ['DCM', 'HCM', 'NOR']
rng = np.random.default_rng(SEED)

folds = pd.read_csv(f'{DRIVE}/folds.csv')
order = folds.patient.values
classes = np.array(sorted(folds.label.unique()))
y_acdc = np.searchsorted(classes, folds.label.values)
rf = pd.read_csv(f'{DRIVE}/step3_challenge_baseline/rich_features.csv')
FEATS = [c for c in rf.columns if c not in ('patient', 'source')]
A_gt = rf[rf.source == 'gt'].set_index('patient').loc[order, FEATS].values
grid = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced')),
                    {'logisticregression__C': [0.01, 0.1, 1, 10, 100]},
                    cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy').fit(A_gt, y_acdc)
head, C_best = grid.best_estimator_, grid.best_params_['logisticregression__C']
pidx = np.searchsorted(classes, PRIMARY)
F = {c: j for j, c in enumerate(FEATS)}

R = pd.DataFrame(list(np.load(f'{OUT}/mnms_concepts.npz', allow_pickle=True)['res']))
Cp = np.stack(list(R['pred']))
def swap_phases(X):
    X = X.copy()
    for a in ('RV', 'MYO', 'LV'):
        X[..., [F[f'{a}_ED'], F[f'{a}_ES']]] = X[..., [F[f'{a}_ES'], F[f'{a}_ED']]]
    for s in ('max', 'mean', 'std', 'min'):
        X[..., [F[f'WT_ED_{s}'], F[f'WT_ES_{s}']]] = X[..., [F[f'WT_ES_{s}'], F[f'WT_ED_{s}']]]
    e = 1e-6; g = lambda c: X[..., F[c]]
    X[..., F['LV_EF']] = 100 * (g('LV_ED') - g('LV_ES')) / np.maximum(g('LV_ED'), e)
    X[..., F['RV_EF']] = 100 * (g('RV_ED') - g('RV_ES')) / np.maximum(g('RV_ED'), e)
    X[..., F['LV_mass']] = 1.05 * g('MYO_ED')
    X[..., F['RV_LV_ED']] = g('RV_ED') / np.maximum(g('LV_ED'), e); X[..., F['RV_LV_ES']] = g('RV_ES') / np.maximum(g('LV_ES'), e)
    X[..., F['MYO_LV_ED']] = g('MYO_ED') / np.maximum(g('LV_ED'), e); X[..., F['MYO_LV_ES']] = g('MYO_ES') / np.maximum(g('LV_ES'), e)
    return X
sw = Cp[:, F['LV_ED']] < Cp[:, F['LV_ES']]
Cp[sw] = swap_phases(Cp[sw])
yv = np.searchsorted(PRIMARY, R.label.values); vend = R.vendor.values
L0 = head.decision_function(Cp)[:, pidx]

def fit_offsets(L, yy):
    """A1: class offsets b (3,) on top of fixed logits, L2-penalised towards 0."""
    Y = np.eye(3)[yy]
    def obj(b):
        Z = L + b; P = np.exp(log_softmax(Z, 1))
        return -(Y * log_softmax(Z, 1)).sum() + LAM * (b ** 2).sum(), (P - Y).sum(0) + 2 * LAM * b
    return minimize(obj, np.zeros(3), jac=True, method='L-BFGS-B').x

def refit(idx):
    """A2: head re-fitted on ACDC (reference) + k-shot patients (predicted), weight W_NEW."""
    X = np.vstack([A_gt, Cp[idx]]); yy = np.concatenate([y_acdc, pidx[yv[idx]]])
    w = np.concatenate([np.ones(len(y_acdc)), np.full(len(idx), W_NEW)])
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced', C=C_best))
    m.fit(X, yy, logisticregression__sample_weight=w)
    return m

rows, offs = [], []
for v in sorted(set(vend)):
    iv = np.where(vend == v)[0]
    present = sorted(set(yv[iv]))
    rows.append(dict(vendor=v, k=0, method='A0 locked', bal_acc=balanced_accuracy_score(yv[iv], L0[iv].argmax(1)), sd=0.0, n_test=len(iv)))
    for k in KS:
        if any((yv[iv] == c).sum() <= k for c in present):
            continue
        acc = {'A0 locked': [], 'A1 intercept-only': [], 'A2 joint refit': []}
        for _ in range(DRAWS):
            ad = np.concatenate([rng.choice(iv[yv[iv] == c], k, replace=False) for c in present])
            te = np.setdiff1d(iv, ad)
            acc['A0 locked'].append(balanced_accuracy_score(yv[te], L0[te].argmax(1)))
            b = fit_offsets(L0[ad], yv[ad]); offs.append(dict(vendor=v, k=k, **dict(zip(PRIMARY, b - b.mean()))))
            acc['A1 intercept-only'].append(balanced_accuracy_score(yv[te], (L0[te] + b).argmax(1)))
            m2 = refit(ad)
            acc['A2 joint refit'].append(balanced_accuracy_score(yv[te], m2.decision_function(Cp[te])[:, pidx].argmax(1)))
        for meth, a in acc.items():
            rows.append(dict(vendor=v, k=k, method=meth, bal_acc=np.mean(a), sd=np.std(a), n_test=len(iv) - k * len(present)))
        print(f'{v}: k = {k} done')

T = pd.DataFrame(rows)
pd.set_option('display.width', 200)
print('\nBalanced accuracy on held-out patients of each vendor (mean +- SD over 100 draws):')
print(T.pivot_table(index=['vendor', 'k'], columns='method', values='bal_acc').round(3).to_string())

print('\nPooled over vendors (weighted by test size):')
for k in KS:
    t = T[T.k == k]
    if t.empty: continue
    print(f'  k = {k:2d}: ' + ' | '.join(f'{m}: {np.average(g.bal_acc, weights=g.n_test):.3f}' for m, g in t.groupby('method'))
          + f'  (vendors: {sorted(t.vendor.unique())})')

O = pd.DataFrame(offs)
print('\nHow the rule moved (A1 class offsets, centred, median over draws, k = 5): positive = class made more likely')
print(O[O.k == 5].groupby('vendor')[PRIMARY].median().round(2).to_string())

T.to_csv(f'{OUT}/step10_fewshot.csv', index=False); O.to_csv(f'{OUT}/step10_offsets.csv', index=False)
print('\nSaved to', OUT)


## Part 7 — Ablation and robustness (Sections 3.10 and 4.6)
*Does each choice matter, and do the conclusions hold?* Measurement groups and flag components are ablated and models are compared with paired tests; then three checks answer the main limitations: the choice of the flag threshold, the small number of internal errors (ten repeated splits) and body-size indexing.

### Ablations, McNemar and Wilcoxon (CPU)
Tables 5–6 and the paired tests.

In [ ]:
import os, glob, numpy as np, pandas as pd
from scipy.stats import binomtest, wilcoxon, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
OUT = f'{DRIVE}/step11_ablation'
os.makedirs(OUT, exist_ok=True)
SEED, B, REP = 42, 2000, 20
rng = np.random.default_rng(SEED)
pd.set_option('display.width', 220)

folds = pd.read_csv(f'{DRIVE}/folds.csv')
order = folds.patient.values
classes = np.array(sorted(folds.label.unique()))
y = np.searchsorted(classes, folds.label.values)
f = folds.fold.values
n, K = len(y), len(classes)
rf = pd.read_csv(f'{DRIVE}/step3_challenge_baseline/rich_features.csv')
FEATS = [c for c in rf.columns if c not in ('patient', 'source')]
C_gt = rf[rf.source == 'gt'].set_index('patient').loc[order, FEATS].values
C_pr = rf[rf.source == 'pred'].set_index('patient').loc[order, FEATS].values
CS = np.load(f'{DRIVE}/step8_patient_uncertainty/concept_samples.npz')['CS']
F = {c: j for j, c in enumerate(FEATS)}

def head(cols=None):
    return GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced')),
                        {'logisticregression__C': [0.01, 0.1, 1, 10, 100]},
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy')

def metrics(P, i):
    return np.array([balanced_accuracy_score(y[i], P[i].argmax(1)),
                     roc_auc_score(y[i], P[i], multi_class='ovr', labels=range(K))])
boots = [i for i in (rng.integers(0, n, n) for _ in range(B)) if len(np.unique(y[i])) == K]

GROUPS = {'volumes (6)': [f'{a}_{p}' for a in ('RV', 'MYO', 'LV') for p in ('ED', 'ES')],
          'ejection fractions (2)': ['LV_EF', 'RV_EF'],
          'LV mass (1)': ['LV_mass'],
          'ratios (4)': ['RV_LV_ED', 'RV_LV_ES', 'MYO_LV_ED', 'MYO_LV_ES'],
          'wall thickness (8)': [c for c in FEATS if c.startswith('WT_')]}
def oof(cols):
    idx = [F[c] for c in cols]; P = np.zeros((n, K))
    for k in range(5):
        tr, te = f != k, f == k
        m = head().fit(C_gt[tr][:, idx], y[tr]); P[te] = m.predict_proba(C_pr[te][:, idx])
    return P
P_full = oof(FEATS)
bs_full = np.array([metrics(P_full, i) for i in boots])
rows = [dict(variant='all 21 measurements', n_feat=21, bal_acc=metrics(P_full, np.arange(n))[0],
             AUC=metrics(P_full, np.arange(n))[1], d_bal=0.0, lo=0.0, hi=0.0)]
for g, cols in GROUPS.items():
    keep = [c for c in FEATS if c not in cols]
    P = oof(keep); d = np.array([metrics(P, i) for i in boots]) - bs_full
    pt = metrics(P, np.arange(n))
    rows.append(dict(variant=f'without {g}', n_feat=len(keep), bal_acc=pt[0], AUC=pt[1],
                     d_bal=pt[0] - rows[0]['bal_acc'], lo=np.percentile(d[:, 0], 2.5), hi=np.percentile(d[:, 0], 97.5)))
    print(f'A: without {g} done')
TA = pd.DataFrame(rows)
print('\nA. Measurement-group ablation (ACDC, out-of-fold; d_bal = minus full, paired bootstrap 95% CI):')
print(TA.round(3).to_string(index=False))

heads, wrong = {}, np.zeros(n, bool)
for k in range(5):
    tr, te = np.where(f != k)[0], np.where(f == k)[0]
    heads[k] = head().fit(C_gt[tr], y[tr]).best_estimator_
    wrong[te] = heads[k].predict(C_pr[te]) != y[te]
print(f'\nACDC errors of the main rule: {wrong.sum()}')

def flag_scores(samples, kappa_mode):
    """max flip fraction per patient; kappa fitted per fold on training patients (or 1)."""
    dv = samples - samples.mean(1, keepdims=True); sd = samples.std(1) + 1e-9
    fmax = np.zeros(n)
    for k in range(5):
        tr, te = np.where(f != k)[0], np.where(f == k)[0]
        kap = 1.0 if kappa_mode == 'one' else float(np.median(np.abs(C_pr[tr] - C_gt[tr]) / sd[tr]) / 0.6745)
        m = heads[k]
        for t in te:
            base = m.predict(C_pr[[t]])[0]
            fl = []
            for j in range(len(FEATS)):
                X = np.repeat(C_pr[[t]], samples.shape[1], 0); X[:, j] += kap * dv[t, :, j]
                fl.append(np.mean(m.predict(X) != base))
            fmax[t] = max(fl)
    return fmax

def flag_summary(fmax, w, thr=0.20):
    fl = fmax >= thr
    return dict(AUROC=roc_auc_score(w, fmax), sens=fl[w].mean(), false_flag=fl[~w].mean())

conf_auc = roc_auc_score(wrong, 1 - np.load(f'{DRIVE}/step8_patient_uncertainty/oof_probs_plain.npy').max(1))
rows = []
for S in (5, 10, 20):
    reps = 1 if S == 20 else REP
    res = []
    for _ in range(reps):
        sub = np.stack([CS[i, rng.choice(20, S, replace=False)] for i in range(n)]) if S < 20 else CS
        fm = flag_scores(sub, 'fit')
        rho = np.median([spearmanr(sub.std(1)[:, F[c]], np.abs(C_pr[:, F[c]] - C_gt[:, F[c]]))[0]
                         for c in ('LV_EF', 'RV_EF', 'LV_ED', 'RV_ED', 'MYO_ED', 'RV_LV_ED', 'MYO_LV_ED')])
        res.append(dict(**flag_summary(fm, wrong), median_rho=rho))
    r = pd.DataFrame(res)
    rows.append(dict(S=S, repeats=reps, **{c: r[c].mean() for c in r}, AUROC_sd=r.AUROC.std() if reps > 1 else 0.0))
    print(f'B: S = {S} done')
TB = pd.DataFrame(rows)
print(f'\nB. Number of samples S (ACDC; flag threshold 0.20; confidence AUROC = {conf_auc:.3f}):')
print(TB.round(3).to_string(index=False))

fm20 = flag_scores(CS, 'fit'); fm1 = flag_scores(CS, 'one')
TD = pd.DataFrame([dict(kappa='fitted (main)', **flag_summary(fm20, wrong)), dict(kappa='1 (no inflation)', **flag_summary(fm1, wrong))])
print('\nD. Inflation factor (ACDC flag, S = 20):'); print(TD.round(3).to_string(index=False))

ext = pd.read_csv(f'{DRIVE}/step9_mnms_external/step9c_per_patient.csv')
w_ext = (ext.type != 'CORRECT').values; seg = (ext.type == 'SEG').values; ok = (ext.type == 'CORRECT').values
rows = []
for thr in (0.10, 0.20, 0.30, 0.40):
    a = flag_summary(fm20, wrong, thr); fe = ext.flip_max.values >= thr
    rows.append(dict(threshold=thr, ACDC_sens=a['sens'], ACDC_false_flag=a['false_flag'],
                     MnMs_sens_all=fe[w_ext].mean(), MnMs_sens_SEG=fe[seg].mean(), MnMs_false_flag=fe[ok].mean()))
TC = pd.DataFrame(rows)
print('\nC. Flag threshold (main = 0.20):'); print(TC.round(3).to_string(index=False))

P = {'logistic rule (main)': np.load(f'{DRIVE}/step2b_concepts21/oof_probs_concept21_LR_pred.npy'),
     'random forest': np.load(f'{DRIVE}/step3_challenge_baseline/oof_probs_challenge_RF_pred.npy')}
for name, pat in (('CineMA linear probe', f'{DRIVE}/cinema_oof_probs.npy'),
                  ('additive model', f'{DRIVE}/step5_c1_additive/oof_probs_additive.npy'),
                  ('per-patient uncertainty (pp_cal)', f'{DRIVE}/step8_patient_uncertainty/oof_probs_pp_cal.npy')):
    if os.path.exists(pat): P[name] = np.load(pat)
j4 = sorted(glob.glob(f'{DRIVE}/step4_joint/*oof*prob*.npy'))
if j4: P['joint model'] = np.load(j4[0]); print('joint model probabilities:', os.path.basename(j4[0]))

def brier(Pm): return ((Pm - np.eye(K)[y]) ** 2).sum(1)
def nll(Pm): return -np.log(np.clip(Pm[np.arange(n), y], 1e-12, 1))
ref = P['logistic rule (main)']; c_ref = ref.argmax(1) == y
rows = []
for name, Pm in P.items():
    if name == 'logistic rule (main)': continue
    c = Pm.argmax(1) == y
    b01, b10 = int((c_ref & ~c).sum()), int((~c_ref & c).sum())
    p_mc = binomtest(b01, b01 + b10, 0.5).pvalue if b01 + b10 > 0 else 1.0
    wb, wn = wilcoxon(brier(ref), brier(Pm)), wilcoxon(nll(ref), nll(Pm))
    rows.append(dict(comparison=f'main vs {name}', acc_main=c_ref.mean(), acc_other=c.mean(), main_only_right=b01,
                     other_only_right=b10, McNemar_p=p_mc, median_dBrier=np.median(brier(ref) - brier(Pm)),
                     Wilcoxon_Brier_p=wb.pvalue, median_dNLL=np.median(nll(ref) - nll(Pm)), Wilcoxon_NLL_p=wn.pvalue))
TE = pd.DataFrame(rows)
m = len(TE); o = np.argsort(TE.McNemar_p.values); adj = np.zeros(m)
for r_, i in enumerate(o):
    adj[i] = min(1.0, max(adj[o[r_ - 1]] if r_ else 0, (m - r_) * TE.McNemar_p.values[i]))
TE['McNemar_p_Holm'] = adj
print('\nE. Paired tests against the main rule (ACDC, n = 100). dBrier/dNLL = main minus other (negative = main better):')
print(TE.round(4).to_string(index=False))

for name, T in (('A_groups', TA), ('B_samples', TB), ('C_threshold', TC), ('D_kappa', TD), ('E_paired_tests', TE)):
    T.to_csv(f'{OUT}/{name}.csv', index=False)
print('\nSaved to', OUT)


### Flag threshold (CPU)
Full threshold curve and nested choice.

In [ ]:
E = ext_load(); folds = E['folds']; FLAG = 0.20
ex = pd.read_csv(f"{os.environ['ACDC_DRIVE']}/step8_patient_uncertainty/doubt_per_patient.csv").set_index('patient').loc[E['order']]
fa, ea = ex.doubt1_flip.values, (ex.label != ex.pred).values
fm, tm = ext_flip_max(E), E['typ']
GRID = np.round(np.arange(0.05, 0.501, 0.05), 2)
def curve(f, e):
    return pd.DataFrame([dict(threshold=t, sensitivity=np.mean(f[e] >= t), false_flag=np.mean(f[~e] >= t),
                              youden=np.mean(f[e] >= t) - np.mean(f[~e] >= t)) for t in GRID])
print('ACDC (9 errors, 91 correct):'); print(curve(fa, ea).round(2).to_string(index=False))
m = (tm == 'SEG') | (tm == 'CORRECT')
print('\nM&Ms-1, segmentation-driven errors vs correct:'); print(curve(fm[m], (tm == 'SEG')[m]).round(2).to_string(index=False))
chosen, hit = [], np.zeros(len(fa), bool)
for k in sorted(folds.fold.unique()):
    te = folds.fold.values == k
    c = curve(fa[~te], ea[~te]); best = c.threshold[c.youden == c.youden.max()].min()
    chosen.append(best); hit[te] = fa[te] >= best
print(f'\nNested choice: thresholds {chosen}; out-of-fold sensitivity {hit[ea].mean():.2f}, false-flag {hit[~ea].mean():.2f}')
print(f'Fixed 0.20: sensitivity {np.mean(fa[ea] >= FLAG):.2f}, false-flag {np.mean(fa[~ea] >= FLAG):.2f}')


### Repeated cross-validation of the flag (CPU)
Ten 5-fold splits instead of one.

In [ ]:
E = ext_load(); C_gt, C_pr, y = E['A_gt'], E['A_pr'], E['y']
dev, sd = E['A_cs'] - E['A_cs'].mean(1, keepdims=True), E['A_cs'].std(1) + 1e-9
n, D, S, FLAG = C_pr.shape[0], C_pr.shape[1], 20, 0.20
rows = []
for seed in range(10):
    pred, fmax, conf = np.zeros(n, int), np.zeros(n), np.zeros(n)
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(C_pr, y):
        mdl = ext_fit(C_gt[tr], y[tr], seed)
        kap = float(np.median(np.abs(C_pr[tr] - C_gt[tr]) / sd[tr]) / 0.6745)
        P = mdl.predict_proba(C_pr[te]); pred[te] = P.argmax(1); conf[te] = 1 - P.max(1)
        for t in te:
            for j in range(D):
                X = np.repeat(C_pr[[t]], S, 0); X[:, j] += kap * dev[t, :, j]
                fmax[t] = max(fmax[t], np.mean(mdl.predict(X) != pred[t]))
    err = pred != y
    rows.append(dict(seed=seed, bal_acc=balanced_accuracy_score(y, pred), errors=err.sum(),
                     AUROC_flag=roc_auc_score(err, fmax), AUROC_conf=roc_auc_score(err, conf),
                     sens_020=np.mean(fmax[err] >= FLAG), false_flag_020=np.mean(fmax[~err] >= FLAG)))
T = pd.DataFrame(rows)
print(T.round(3).to_string(index=False))
print(pd.DataFrame({'mean': T.drop(columns='seed').mean(), 'sd': T.drop(columns='seed').std()}).round(3).to_string())
print(f'{T.errors.sum()} error instances; flag better than confidence in {np.mean(T.AUROC_flag > T.AUROC_conf) * 100:.0f} % of splits')
T.to_csv(f"{os.environ['ACDC_DRIVE']}/step14_robustness/step14_repeated_cv.csv", index=False)


### BSA indexing (CPU)
Indexed vs non-indexed rule on ACDC and on the M&Ms-1 patients with height and weight.

In [ ]:
E = ext_load(); FI, folds, y = E['F'], E['folds'], E['y']
IDX = [FI[c] for c in ('RV_ED', 'RV_ES', 'MYO_ED', 'MYO_ES', 'LV_ED', 'LV_ES', 'LV_mass')]
bsa = lambda h_cm, w_kg: np.sqrt(h_cm * w_kg / 3600.0)
def index(X, b):
    X = X.copy(); X[:, IDX] = X[:, IDX] / b[:, None]; return X
hw = pd.read_csv(f"{os.environ['ACDC_DRIVE']}/step14_robustness/acdc_height_weight.csv").set_index('patient').loc[E['order']]
B_acdc = bsa(hw.height.values, hw.weight.values)
pr_raw, pr_idx = np.zeros(len(y), int), np.zeros(len(y), int)
for k in sorted(folds.fold.unique()):
    tr, te = folds.fold.values != k, folds.fold.values == k
    pr_raw[te] = ext_fit(E['A_gt'][tr], y[tr]).predict(E['A_pr'][te])
    pr_idx[te] = ext_fit(index(E['A_gt'][tr], B_acdc[tr]), y[tr]).predict(index(E['A_pr'][te], B_acdc[te]))
print(f'ACDC out-of-fold balanced accuracy: non-indexed {balanced_accuracy_score(y, pr_raw):.2f} | indexed {balanced_accuracy_score(y, pr_idx):.2f}')
mi = E['meta'].set_index('External code')
h = pd.to_numeric(E['R'].code.map(mi['Height']), errors='coerce').values
w = pd.to_numeric(E['R'].code.map(mi['Weight']), errors='coerce').values
if np.nanmedian(h) < 3: h = h * 100
has = (h > 0) & (w > 0); b = bsa(h[has], w[has])
print(f'M&Ms-1 primary patients with height and weight: {has.sum()}; median BSA {np.median(b):.2f} vs ACDC {np.median(B_acdc):.2f} m2')
head_idx = ext_fit(index(E['A_gt'], B_acdc), y)
def dec3(model, X): return model.decision_function(X)[:, E['pidx']].argmax(1)
rows, yy = [], E['yv'][has]
for name, model, Xp, Xg in [('non-indexed (locked)', E['head'], E['Cp'][has], E['Cg'][has]),
                            ('BSA-indexed', head_idx, index(E['Cp'][has], b), index(E['Cg'][has], b))]:
    p, r_ = dec3(model, Xp), dec3(model, Xg); wr = p != yy
    rows.append(dict(rule=name, n=has.sum(), bal_acc=balanced_accuracy_score(yy, p), bal_acc_refmeas=balanced_accuracy_score(yy, r_),
                     errors=wr.sum(), SEG=np.sum(wr & (r_ == yy)), RULE=np.sum(wr & (r_ != yy))))
print(pd.DataFrame(rows).round(2).to_string(index=False))


## Part 8 — Ideas that did not work (Section 4.7)
*Two promising ideas, tested with pre-specified checks.* Physiological conservation laws as a label-free adaptation signal, and one shared Gaussian error model; their failure motivated the per-patient scheme of Part 4.

### Physiological laws (CPU)
Negative result.

In [ ]:
import os, numpy as np, pandas as pd
from scipy.stats import spearmanr, wilcoxon

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
OUT = f'{DRIVE}/step6_feasibility'
os.makedirs(OUT, exist_ok=True)

folds = pd.read_csv(f'{DRIVE}/folds.csv')
rf = pd.read_csv(f'{DRIVE}/step3_challenge_baseline/rich_features.csv')
G = rf[rf.source == 'gt'].set_index('patient').loc[folds.patient]
P = rf[rf.source == 'pred'].set_index('patient').loc[folds.patient]
lab = folds.label.values

def laws(D):
    r1 = 100 * (D.MYO_ES / D.MYO_ED - 1)
    r2 = (D.LV_ED - D.LV_ES) - (D.RV_ED - D.RV_ES)
    return r1.values, r2.values

g1, g2 = laws(G)
p1, p2 = laws(P)

rows = []
for c in ['ALL'] + sorted(set(lab)):
    m = np.ones(len(lab), bool) if c == 'ALL' else lab == c
    rows.append(dict(cls=c, n=m.sum(),
                     MYO_gt_mean=g1[m].mean(), MYO_gt_absmed=np.median(abs(g1[m])),
                     MYO_pred_mean=p1[m].mean(), MYO_pred_absmed=np.median(abs(p1[m])),
                     SV_gt_mean=g2[m].mean(), SV_gt_absmed=np.median(abs(g2[m])),
                     SV_pred_mean=p2[m].mean(), SV_pred_absmed=np.median(abs(p2[m]))))
tab = pd.DataFrame(rows)
print('Law violations (MYO in %, SV in mL); absmed = median of |violation|')
print(tab.round(1).to_string(index=False))

for name, a, b in (('MYO law', p1, g1), ('SV law', p2, g2)):
    s, pv = wilcoxon(abs(a), abs(b))
    print(f'\n{name}: |violation| pred > gt in {np.mean(abs(a) > abs(b)) * 100:.0f}% of patients, Wilcoxon p = {pv:.2g}')

e_myo = abs(P.MYO_ES.values - G.MYO_ES.values) + abs(P.MYO_ED.values - G.MYO_ED.values)
e_sv = abs((P.LV_ED - P.LV_ES).values - (G.LV_ED - G.LV_ES).values) + abs((P.RV_ED - P.RV_ES).values - (G.RV_ED - G.RV_ES).values)
e_ef = abs(P.LV_EF.values - G.LV_EF.values) + abs(P.RV_EF.values - G.RV_EF.values)
print('\nSpearman: |violation on pred|  vs  measurement error (pred vs gt)')
for name, v in (('MYO law', abs(p1)), ('SV law', abs(p2))):
    for en, e in (('MYO vol error', e_myo), ('SV error', e_sv), ('EF error', e_ef)):
        r, pv = spearmanr(v, e)
        print(f'  {name:8s} vs {en:14s}: rho = {r:+.2f}  p = {pv:.2g}')

for name, a, b in (('MYO law', p1, g1), ('SV law', p2, g2)):
    print(f'\n{name}: var(gt) = {b.var():.1f}, var(pred) = {a.var():.1f}, '
          f'var(pred - gt) = {(a - b).var():.1f}  -> seg-induced share ≈ {(a - b).var() / a.var() * 100:.0f}%')

tab.to_csv(f'{OUT}/law_violations_by_class.csv', index=False)
pd.DataFrame(dict(patient=folds.patient, label=lab, myo_gt=g1, myo_pred=p1, sv_gt=g2, sv_pred=p2,
                  err_myo=e_myo, err_sv=e_sv, err_ef=e_ef)).to_csv(f'{OUT}/per_patient.csv', index=False)
print('\nSaved to', OUT)


### Single Gaussian error model and conformal sets (CPU)
Negative result.

In [ ]:
import os, numpy as np, pandas as pd
from scipy.special import softmax
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
OUT = f'{DRIVE}/step7_eiv_conformal'
os.makedirs(OUT, exist_ok=True)
SEED, ALPHA, MC = 42, 0.10, 4000
rng = np.random.default_rng(SEED)

folds = pd.read_csv(f'{DRIVE}/folds.csv')
order = folds.patient.values
classes = np.array(sorted(folds.label.unique()))
y = np.searchsorted(classes, folds.label.values)
f = folds.fold.values
n, K = len(y), len(classes)
rf = pd.read_csv(f'{DRIVE}/step3_challenge_baseline/rich_features.csv')
FEATS = [c for c in rf.columns if c not in ('patient', 'source')]
C_gt = rf[rf.source == 'gt'].set_index('patient').loc[order, FEATS].values
C_pr = rf[rf.source == 'pred'].set_index('patient').loc[order, FEATS].values
D = len(FEATS)
print(f'{n} patients | {D} concepts | alpha = {ALPHA}')

def fit_head(idx):
    g = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000, class_weight='balanced')),
                     {'logisticregression__C': [0.01, 0.1, 1, 10, 100]},
                     cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='balanced_accuracy')
    g.fit(C_gt[idx], y[idx])
    sc, lr = g.best_estimator_.named_steps['standardscaler'], g.best_estimator_.named_steps['logisticregression']
    return sc, lr.coef_, lr.intercept_

def error_model(idx):
    """bias and covariance of (pred - gt) measurement error, from these patients only."""
    E = C_pr[idx] - C_gt[idx]
    return E.mean(0), LedoitWolf().fit(E).covariance_

def predict(sc, W, b, bias, Sig, idx, mode):
    """mode: plain | var (propagate variance) | full (bias correction + variance)."""
    c = C_pr[idx] - (bias if mode == 'full' else 0)
    mu = ((c - sc.mean_) / sc.scale_) @ W.T + b
    if mode == 'plain':
        return softmax(mu, 1), np.zeros((len(idx), K, K))
    Ws = W / sc.scale_
    Sz = Ws @ Sig @ Ws.T
    L = np.linalg.cholesky(Sz + 1e-9 * np.eye(K))
    eps = rng.standard_normal((MC, K)) @ L.T
    P = np.stack([softmax(m + eps, 1).mean(0) for m in mu])
    return P, np.broadcast_to(Sz, (len(idx), K, K))

def score(P, yy):
    return 1 - P[np.arange(len(yy)), yy]

def aps_score(P, yy):
    o = np.argsort(-P, 1); cs = np.cumsum(np.take_along_axis(P, o, 1), 1)
    r = np.argmax(o == yy[:, None], 1)
    return cs[np.arange(len(yy)), r]

def qhat(s):
    m = len(s); return np.quantile(s, min(1, np.ceil((m + 1) * (1 - ALPHA)) / m), method='higher')

MODES = ['plain', 'var', 'full']
P = {m: np.zeros((n, K)) for m in MODES}
S_lac = {m: np.zeros((n, K), bool) for m in MODES}
S_aps = {m: np.zeros((n, K), bool) for m in MODES}
expl = []
for k in range(5):
    tr, te = np.where(f != k)[0], np.where(f == k)[0]
    cal = {m: [np.zeros(len(tr)), np.zeros(len(tr))] for m in MODES}
    for a, bb in StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, y[tr]):
        sc, W, b0 = fit_head(tr[a]); bias, Sig = error_model(tr[a])
        for m in MODES:
            Pc, _ = predict(sc, W, b0, bias, Sig, tr[bb], m)
            cal[m][0][bb], cal[m][1][bb] = score(Pc, y[tr[bb]]), aps_score(Pc, y[tr[bb]])
    sc, W, b0 = fit_head(tr); bias, Sig = error_model(tr)
    for m in MODES:
        Pt, _ = predict(sc, W, b0, bias, Sig, te, m)
        P[m][te] = Pt
        q1, q2 = qhat(cal[m][0]), qhat(cal[m][1])
        S_lac[m][te] = (1 - Pt) <= q1
        o = np.argsort(-Pt, 1); cs = np.cumsum(np.take_along_axis(Pt, o, 1), 1)
        keep = np.zeros_like(Pt, bool)
        for i in range(len(te)):
            j = np.searchsorted(cs[i], q2 - 1e-12) + 1
            keep[i, o[i, :min(j, K)]] = True
        S_aps[m][te] = keep
    Ws = W / sc.scale_
    for t in te:
        Pt = P['full'][t]; a1, a2 = np.argsort(-Pt)[:2]
        d = Ws[a1] - Ws[a2]
        var = d @ Sig @ d
        margin = (((C_pr[t] - bias - sc.mean_) / sc.scale_) @ (W[a1] - W[a2])) + b0[a1] - b0[a2]
        flip = np.abs(margin / np.where(np.abs(d) > 1e-12, d, np.nan)) / np.sqrt(np.diag(Sig))
        top = np.argsort(np.nan_to_num(flip, nan=np.inf))[:3]
        expl.append(dict(patient=order[t], label=classes[y[t]], top1=classes[a1], top2=classes[a2],
                         margin=margin, margin_sd=np.sqrt(var), z=margin / np.sqrt(var),
                         **{f'doubt{r+1}': FEATS[j] for r, j in enumerate(top)},
                         **{f'doubt{r+1}_flipSD': flip[j] for r, j in enumerate(top)}))
    print(f'fold {k} done')

def ece(Pm, bins=15):
    conf, acc = Pm.max(1), (Pm.argmax(1) == y).astype(float)
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

rows = []
for m in MODES:
    for nm, S in (('LAC', S_lac[m]), ('APS', S_aps[m])):
        cov = S[np.arange(n), y]
        rows.append(dict(model=m, sets=nm, bal_acc=balanced_accuracy_score(y, P[m].argmax(1)),
                         AUC=roc_auc_score(y, P[m], multi_class='ovr'), ECE=ece(P[m]),
                         NLL=-np.mean(np.log(np.clip(P[m][np.arange(n), y], 1e-12, 1))),
                         coverage=cov.mean(), mean_size=S.sum(1).mean(), singleton=(S.sum(1) == 1).mean(),
                         **{f'cov_{c}': cov[y == i].mean() for i, c in enumerate(classes)}))
res = pd.DataFrame(rows)
pd.set_option('display.width', 200)
print(f'\nTarget coverage = {1 - ALPHA:.2f}')
print(res.round(3).to_string(index=False))

ex = pd.DataFrame(expl)
print('\nMost frequent source of doubt (top-1 measurement), full model:')
print(ex.doubt1.value_counts().head(8).to_string())
wrong = ex[ex.label != ex.top1]
print(f'\nMisdiagnosed patients ({len(wrong)}): was the truth the 2nd choice? {np.mean(wrong.label == wrong.top2) * 100:.0f}%')
print(wrong[['patient', 'label', 'top1', 'top2', 'z', 'doubt1', 'doubt1_flipSD']].round(2).to_string(index=False))
print('\n|z| (margin / its SD) for correct vs wrong:',
      f'{ex[ex.label == ex.top1].z.abs().median():.2f} vs {wrong.z.abs().median():.2f}')

res.to_csv(f'{OUT}/step7_results.csv', index=False)
ex.to_csv(f'{OUT}/doubt_explanations.csv', index=False)
for m in MODES:
    np.save(f'{OUT}/oof_probs_{m}.npy', P[m]); np.save(f'{OUT}/sets_LAC_{m}.npy', S_lac[m])
print('\nSaved to', OUT)


## Part 9 — Figures (Figures 2–5)
Figure 1 is a diagram. Files: Figure 2 = `fig3_uncertainty_vs_error`, Figure 3 = `fig2_flag_roc`, Figure 4 = `fig4_error_attribution`, Figure 5 = `fig5_fewshot` (PDF and PNG, 300 dpi).

In [ ]:
import os, io, glob, zipfile, numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullFormatter, LogLocator
from sklearn.metrics import roc_curve, roc_auc_score
from scipy.stats import spearmanr

DRIVE = os.environ.get('ACDC_DRIVE', '/content/drive/MyDrive/CMR_concept_bottleneck')
OUT = f'{DRIVE}/step12_figures'
os.makedirs(OUT, exist_ok=True)
BLUE, VERM, GREEN, GREY, INK = '#0072B2', '#D55E00', '#009E73', '#8C8C8C', '#222222'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.titlesize': 9, 'axes.labelsize': 8,
                     'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6,
                     'ytick.major.width': 0.6, 'axes.edgecolor': '#444444', 'axes.labelcolor': INK,
                     'xtick.color': '#444444', 'ytick.color': '#444444', 'legend.frameon': False,
                     'savefig.dpi': 300, 'savefig.bbox': 'tight', 'pdf.fonttype': 42})
def save(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(f'{OUT}/{name}.{ext}')
    plt.close(fig); print('saved', name)
def panel(ax, letter):
    t = ax.get_title(); ax.set_title('')
    ax.set_title(r'$\bf{' + letter + '}$   ' + t, loc='left')

folds = pd.read_csv(f'{DRIVE}/folds.csv')
order = folds.patient.values
classes = np.array(sorted(folds.label.unique()))
y = np.searchsorted(classes, folds.label.values)
rf = pd.read_csv(f'{DRIVE}/step3_challenge_baseline/rich_features.csv')
FEATS = [c for c in rf.columns if c not in ('patient', 'source')]
F = {c: j for j, c in enumerate(FEATS)}
A_gt = rf[rf.source == 'gt'].set_index('patient').loc[order, FEATS].values
A_pr = rf[rf.source == 'pred'].set_index('patient').loc[order, FEATS].values
A_cs = np.load(f'{DRIVE}/step8_patient_uncertainty/concept_samples.npz')['CS']

def swap_phases(X):
    X = X.copy()
    for a in ('RV', 'MYO', 'LV'):
        X[..., [F[f'{a}_ED'], F[f'{a}_ES']]] = X[..., [F[f'{a}_ES'], F[f'{a}_ED']]]
    for s in ('max', 'mean', 'std', 'min'):
        X[..., [F[f'WT_ED_{s}'], F[f'WT_ES_{s}']]] = X[..., [F[f'WT_ES_{s}'], F[f'WT_ED_{s}']]]
    e = 1e-6; g = lambda c: X[..., F[c]]
    X[..., F['LV_EF']] = 100 * (g('LV_ED') - g('LV_ES')) / np.maximum(g('LV_ED'), e)
    X[..., F['RV_EF']] = 100 * (g('RV_ED') - g('RV_ES')) / np.maximum(g('RV_ED'), e)
    X[..., F['LV_mass']] = 1.05 * g('MYO_ED')
    X[..., F['RV_LV_ED']] = g('RV_ED') / np.maximum(g('LV_ED'), e); X[..., F['RV_LV_ES']] = g('RV_ES') / np.maximum(g('LV_ES'), e)
    X[..., F['MYO_LV_ED']] = g('MYO_ED') / np.maximum(g('LV_ED'), e); X[..., F['MYO_LV_ES']] = g('MYO_ES') / np.maximum(g('LV_ES'), e)
    return X
R = pd.DataFrame(list(np.load(f'{DRIVE}/step9_mnms_external/mnms_concepts.npz', allow_pickle=True)['res']))
M_gt, M_pr, M_cs = np.stack(list(R['gt'])), np.stack(list(R['pred'])), np.stack(list(R['samples']))
sw = M_pr[:, F['LV_ED']] < M_pr[:, F['LV_ES']]
M_gt[sw], M_pr[sw], M_cs[sw] = swap_phases(M_gt[sw]), swap_phases(M_pr[sw]), swap_phases(M_cs[sw])
ext = pd.read_csv(f'{DRIVE}/step9_mnms_external/step9c_per_patient.csv')
R['centre'] = ext.set_index('code').loc[R.code, 'centre'].values

dp = pd.read_csv(f'{DRIVE}/step8_patient_uncertainty/doubt_per_patient.csv').set_index('patient').loc[order]
wrong = (dp.label != dp.pred).values
flip = dp.doubt1_flip.values
conf = 1 - np.load(f'{DRIVE}/step8_patient_uncertainty/oof_probs_plain.npy').max(1)
fig, axs = plt.subplots(1, 2, figsize=(6.6, 3.8))
ax = axs[0]
for s, c, ls, lab in ((flip, BLUE, '-', 'Flip flag'), (conf, GREY, '--', 'Softmax confidence')):
    fpr, tpr, _ = roc_curve(wrong, s)
    ax.plot(fpr, tpr, color=c, ls=ls, lw=1.6, drawstyle='steps-post', label=f'{lab} (AUROC {roc_auc_score(wrong, s):.2f})')
thr = flip >= 0.20
ax.plot(thr[~wrong].mean(), thr[wrong].mean(), 'o', ms=6, mfc='white', mec=BLUE, mew=1.4, label='Flag threshold 0.20')
ax.plot([0, 1], [0, 1], color='#CCCCCC', lw=0.8, zorder=0)
ax.set(xlabel='False-positive rate (correct patients flagged)', ylabel='True-positive rate (errors flagged)',
       title=f'ACDC: {wrong.sum()} errors, {(~wrong).sum()} correct', xlim=(-0.02, 1.02), ylim=(-0.02, 1.02))
ax.set_aspect('equal'); ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2)); panel(ax, 'a')
ax = axs[1]
ok = (ext.type == 'CORRECT').values
for t, c, ls, lab in (('SEG', BLUE, '-', 'Segmentation-driven errors'), ('RULE', VERM, '-.', 'Rule-driven errors')):
    m = (ext.type == t).values; sel = m | ok
    fpr, tpr, _ = roc_curve(m[sel], ext.flip_max.values[sel])
    ax.plot(fpr, tpr, color=c, ls=ls, lw=1.6, drawstyle='steps-post',
            label=f'{lab}, n = {m.sum()} (AUROC {roc_auc_score(m[sel], ext.flip_max.values[sel]):.2f})')
ax.plot([0, 1], [0, 1], color='#CCCCCC', lw=0.8, zorder=0)
ax.set(xlabel='False-positive rate (correct patients flagged)', ylabel='True-positive rate',
       title=f'M&Ms-1: flag by error type ({ok.sum()} correct)', xlim=(-0.02, 1.02), ylim=(-0.02, 1.02))
ax.set_aspect('equal'); ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2)); panel(ax, 'b')
fig.tight_layout(w_pad=2.5); save(fig, 'fig2_flag_roc')

fig, axs = plt.subplots(2, 2, figsize=(6.6, 5.6))
for r_, (feat, unit, name) in enumerate((('LV_EF', 'percentage points', 'LV ejection fraction'), ('RV_ED', 'mL', 'RV end-diastolic volume'))):
    for c_, (lab, gt, pr, cs, col, mk) in enumerate((('ACDC (n = 100)', A_gt, A_pr, A_cs, BLUE, 'o'),
                                                     (f'M&Ms-1 (n = {len(R)})', M_gt, M_pr, M_cs, VERM, 's'))):
        ax = axs[r_, c_]; j = F[feat]
        sd, err = cs[:, :, j].std(1), np.abs(pr[:, j] - gt[:, j])
        rho, p = spearmanr(sd, err)
        fx, fy = np.percentile(sd[sd > 0], 1), np.percentile(err[err > 0], 1)
        ax.scatter(np.maximum(sd, fx), np.maximum(err, fy), s=10, color=col, marker=mk, alpha=0.55, linewidths=0)
        ax.set_xscale('log'); ax.set_yscale('log')
        for a_ in (ax.xaxis, ax.yaxis):
            a_.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
            a_.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}')); a_.set_minor_formatter(NullFormatter())
        ax.set(xlabel=f'Per-patient sample SD ({unit})', ylabel=f'|Predicted − reference| ({unit})', title=f'{name}, {lab}')
        ax.text(0.03, 0.95, f'Spearman ρ = {rho:.2f}, p = {p:.1g}', transform=ax.transAxes, va='top', fontsize=7, color=INK, bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#CCCCCC', lw=0.5))
        panel(ax, 'abcd'[2 * r_ + c_])
fig.tight_layout(h_pad=2.0, w_pad=2.5); save(fig, 'fig3_uncertainty_vs_error')

fig, axs = plt.subplots(1, 2, figsize=(7.0, 3.0), gridspec_kw={'width_ratios': [1, 1.1]})
ax = axs[0]
vend = sorted(ext.vendor.unique(), key=lambda v: -(ext.vendor == v).sum())
cnt = pd.crosstab(ext.vendor, ext.type).reindex(vend).fillna(0)
for k in ('CORRECT', 'SEG', 'RULE'):
    if k not in cnt: cnt[k] = 0
prop = cnt[['CORRECT', 'SEG', 'RULE']].div(cnt.sum(axis=1), axis=0)
left = np.zeros(len(vend))
for k, col, hatch, lab in (('CORRECT', '#D9D9D9', None, 'Correct'), ('SEG', BLUE, None, 'Segmentation-driven error'),
                           ('RULE', VERM, '////', 'Rule-driven error')):
    ax.barh(range(len(vend)), prop[k], left=left, color=col, hatch=hatch, edgecolor='white', linewidth=1.0, height=0.62, label=lab)
    left += prop[k].values
ax.set_yticks(range(len(vend)), [f'{v} (n = {int(cnt.loc[v].sum())})' for v in vend]); ax.invert_yaxis()
ax.set(xlabel='Share of patients', xlim=(0, 1), title='Error source by vendor')
ax.legend(loc='upper center', bbox_to_anchor=(0.45, -0.22), ncol=1); panel(ax, 'a')
ax = axs[1]
rng = np.random.default_rng(0)
labs = ['DCM', 'HCM', 'NOR']
for i, cl in enumerate(labs):
    a = A_gt[classes[y] == cl, F['LV_EF']]
    ax.scatter(i - 0.18 + rng.uniform(-0.07, 0.07, len(a)), a, s=9, color=BLUE, marker='o', alpha=0.7, linewidths=0,
               label='ACDC' if i == 0 else None)
    m = (R.label == cl).values
    for cen3, mfc in ((False, VERM), (True, 'white')):
        mm = m & ((R.centre.values == 3) == cen3)
        b = M_gt[mm, F['LV_EF']]
        if len(b):
            ax.scatter(i + 0.18 + rng.uniform(-0.07, 0.07, len(b)), b, s=9, marker='s', facecolors=mfc, edgecolors=VERM,
                       linewidths=0.6, alpha=0.8, label=('M&Ms-1' if not cen3 else 'M&Ms-1, centre 3') if i == 0 else None)
    for x0, v in ((i - 0.18, a), (i + 0.18, M_gt[m, F['LV_EF']])):
        ax.plot([x0 - 0.1, x0 + 0.1], [np.median(v)] * 2, color=INK, lw=1.4)
ax.set_xticks(range(3), labs); ax.set(ylabel='Reference LV ejection fraction (%)', title='Same label, different patients')
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), handletextpad=0.2); panel(ax, 'b')
fig.tight_layout(w_pad=4.0); save(fig, 'fig4_error_attribution')

T = pd.read_csv(f'{DRIVE}/step9_mnms_external/step10_fewshot.csv')
vs = [v for v in ('Siemens', 'Philips', 'GE', 'Canon') if v in set(T.vendor)]
fig, axs = plt.subplots(1, len(vs), figsize=(6.6, 2.4), sharey=True)
for ax, v in zip(np.atleast_1d(axs), vs):
    t = T[T.vendor == v]
    a0 = t[(t.method == 'A0 locked') & (t.k == 0)].bal_acc.values[0]
    ax.axhline(a0, color=GREY, ls='--', lw=1.0, label='A0 locked')
    for meth, col, mk, ls, lab in (('A1 intercept-only', BLUE, 'o', '-', 'A1 offsets only'), ('A2 joint refit', VERM, 's', '-.', 'A2 full refit')):
        s = t[t.method == meth].sort_values('k')
        ax.fill_between(s.k, s.bal_acc - s.sd, s.bal_acc + s.sd, color=col, alpha=0.12, linewidth=0)
        ax.plot(s.k, s.bal_acc, color=col, marker=mk, ms=4, ls=ls, lw=1.4, label=lab)
    ax.set(title=f'{v} (n = {int(t[t.k == 0].n_test.values[0])})', xticks=sorted(t[t.k > 0].k.unique()))
axs[0].set_ylabel('Balanced accuracy (held-out)')
fig.supxlabel('Labelled patients per class from the new vendor (k)', fontsize=8, y=0.02)
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.1))
fig.tight_layout(w_pad=1.2, rect=(0, 0.04, 1, 1)); save(fig, 'fig5_fewshot')
print('All figures saved to', OUT)
